In [1]:
# 08 · Başlangıç: agent'ı src/'dan yükle, sürüm kontrolü, ısınma
%load_ext autoreload
%autoreload 2
import sys
sys.path.insert(0, "..")

from src import ayarlar, rag_araci
from src.talimat import SISTEM_TALIMATI
from src.agent_graf import calistir, VARSAYILAN_MODEL

print("Talimat uzunluğu:", len(SISTEM_TALIMATI), "(v4 = 3702)")
print("METIN_MODU:", ayarlar.METIN_MODU)
print("Model:", VARSAYILAN_MODEL)
print(f"Isınma: {rag_araci.isinma():.1f} sn")

s = calistir("Elden Ring'in puanı kaç?", oturum="08-kontrol")
print("\nCevap:", s["cevap"])
print("Araçlar:", s["araclar"])
print(f'{s["tur"]} tur · {s["sure"]} sn · girdi {s["girdi_token"]:,} / çıktı {s["cikti_token"]:,} token')

Talimat uzunluğu: 3702 (v4 = 3702)
METIN_MODU: gizle
Model: gpt-4.1-mini
Isınma: 24.1 sn

Cevap: Elden Ring'in Steam oyunları arasında puanı %92,6'dır. Bu bilgi 977.477 oy sayısına dayanıyor, yani oldukça güvenilir bir puandır.
Araçlar: ['oyun_bul({"isim":"Elden Ring"})']
2 tur · 4.1 sn · girdi 8,598 / çıktı 58 token


## 08 · Değerlendirme — Başlangıç
- Agent: `src/agent_graf.py` (LangGraph sürümü, API'de kullanılacak olan), model `gpt-4.1-mini`.
- Sürüm kontrolü: talimat 3.702 karakter = v4 ✓, METIN_MODU = gizle.
- Isınma (embedding GPU'ya, Chroma RAM'e): 24,1 sn. Ölçümden önce şart, yoksa ilk soru süresi bozulur.
- Deneme: "Elden Ring'in puanı kaç?" → oyun_bul → %92,6 (977.477 oy), 2 tur, 4,1 sn, 8.598 girdi token.

In [2]:
# 08 · Soru seti 1/6: Sayı / SQL soruları + referans cevaplar (doğrudan steam.db'den)
import pandas as pd
from contextlib import closing
from src.sql_araci import baglan

pd.set_option("display.max_colwidth", None)
SORULAR = []

def ekle(kimlik, kategori, soru, beklenen_arac, kontrol, ref_sql=None):
    SORULAR.append({"id": kimlik, "kategori": kategori, "soru": soru,
                    "beklenen_arac": beklenen_arac, "kontrol": kontrol, "ref_sql": ref_sql})

def referans(sql):
    with closing(baglan()) as b:
        return pd.read_sql(sql, b)

ekle("S1", "sql", "Baldur's Gate 3'ün Steam puanı ve oy sayısı nedir?",
     ["oyun_bul"], "olumlu_oran ve oy_sayisi referansla aynı olmalı",
     "SELECT app_id, ad, olumlu_oran, oy_sayisi FROM oyunlar WHERE ad = 'Baldur''s Gate 3'")

ekle("S2", "sql", "100 binden fazla oy almış kaç oyun var?",
     ["sql_calistir"], "sayı referansla aynı olmalı (COUNT ile)",
     "SELECT COUNT(*) AS oyun_sayisi FROM oyunlar WHERE oy_sayisi > 100000")

ekle("S3", "sql", "Türkçe destekleyen en yüksek puanlı korku oyunları hangileri?",
     ["sql_calistir"], "oy ≥ 1000 + orana göre sıralama; ilk oyunlar referansla örtüşmeli",
     """SELECT o.app_id, o.ad, o.olumlu_oran, o.oy_sayisi, o.dil_spam
        FROM oyunlar o
        WHERE o.oy_sayisi >= 1000
          AND EXISTS (SELECT 1 FROM oyun_etiket e WHERE e.app_id = o.app_id AND e.etiket = 'Horror')
          AND EXISTS (SELECT 1 FROM oyun_dil d WHERE d.app_id = o.app_id AND d.dil = 'Turkish')
        ORDER BY o.olumlu_oran DESC LIMIT 10""")

ekle("S4", "sql", "Cyberpunk 2077 kaç dolar?",
     ["oyun_bul", "sql_calistir"], "normal_fiyat kullanmalı, birim $ olmalı",
     "SELECT app_id, ad, fiyat, normal_fiyat, indirim_yuzde FROM oyunlar WHERE ad = 'Cyberpunk 2077'")

ekle("S5", "sql", "En çok Türkçe yorum alan 5 oyun hangisi?",
     ["sql_calistir"], "tr_yorum_sayisi sütunu (ya da yorumlar.dil = 'tr'); liste referansla aynı olmalı",
     "SELECT app_id, ad, tr_yorum_sayisi FROM oyunlar ORDER BY tr_yorum_sayisi DESC LIMIT 5")

ekle("S6", "sql", "İnternetten arkadaşlarla birlikte oynanabilen kaç oyun var?",
     ["sql_calistir"], "kategori 'Online Co-op' olmalı; sayı referansla aynı",
     "SELECT COUNT(DISTINCT app_id) AS oyun_sayisi FROM oyun_kategori WHERE kategori = 'Online Co-op'")

for s in SORULAR:
    print(f"\n[{s['id']}] {s['soru']}\n  beklenen araç: {s['beklenen_arac']} · kontrol: {s['kontrol']}")
    display(referans(s["ref_sql"]))


[S1] Baldur's Gate 3'ün Steam puanı ve oy sayısı nedir?
  beklenen araç: ['oyun_bul'] · kontrol: olumlu_oran ve oy_sayisi referansla aynı olmalı


,app_id,ad,olumlu_oran,oy_sayisi
0,1086940,Baldur's Gate 3,0.968,669941



[S2] 100 binden fazla oy almış kaç oyun var?
  beklenen araç: ['sql_calistir'] · kontrol: sayı referansla aynı olmalı (COUNT ile)


,oyun_sayisi
0,82



[S3] Türkçe destekleyen en yüksek puanlı korku oyunları hangileri?
  beklenen araç: ['sql_calistir'] · kontrol: oy ≥ 1000 + orana göre sıralama; ilk oyunlar referansla örtüşmeli


,app_id,ad,olumlu_oran,oy_sayisi,dil_spam
0,2070550,A Castle Full of Cats,0.994,3709,0
1,1361320,The Room 4: Old Sins,0.986,15253,0
2,2291850,Underground Blossom,0.984,5713,0
3,546560,Half-Life: Alyx,0.983,92558,0
4,1292940,Cube Escape Collection,0.983,6286,0
5,1999360,Placid Plastic Duck Simulator,0.980,15900,0
6,2763670,Arctic Eggs,0.974,2111,0
7,1281800,Samsara Room,0.973,8712,0
8,1332010,Stray,0.973,146316,0
9,2475490,Mouthwashing,0.973,10171,0


... (2 outputs left out)


[S5] En çok Türkçe yorum alan 5 oyun hangisi?
  beklenen araç: ['sql_calistir'] · kontrol: tr_yorum_sayisi sütunu (ya da yorumlar.dil = 'tr'); liste referansla aynı olmalı


,app_id,ad,tr_yorum_sayisi
0,1293830,Forza Horizon 4,29787
1,261550,Mount & Blade II: Bannerlord,24929
2,239140,Dying Light,12207
3,1238840,Battlefield™ 1,12182
4,1583320,Pro Soccer Online,7947



[S6] İnternetten arkadaşlarla birlikte oynanabilen kaç oyun var?
  beklenen araç: ['sql_calistir'] · kontrol: kategori 'Online Co-op' olmalı; sayı referansla aynı


,oyun_sayisi
0,3895


## Soru seti 1/6 · Sayı / SQL (S1–S6)
Her sorunun referans cevabı agent'a sormadan doğrudan steam.db'den hesaplandı (araç sınırları referansı etkilemesin diye `sql_calistir` değil `pd.read_sql`).

| id | Soru | Beklenen araç | Referans |
|---|---|---|---|
| S1 | Baldur's Gate 3 puanı ve oy sayısı | oyun_bul | %96,8 · 669.941 oy |
| S2 | 100 binden fazla oylu oyun sayısı | sql_calistir (COUNT) | 82 |
| S3 | Türkçe destekli en yüksek puanlı korku | sql_calistir | oy ≥ 1000 + oran: A Castle Full of Cats, The Room 4, Underground Blossom, Half-Life: Alyx... |
| S4 | Cyberpunk 2077 fiyatı | oyun_bul + sql_calistir | normal_fiyat 59,98 $ (o gün %50 indirimle 29,99 $) |
| S5 | En çok Türkçe yorum alan 5 oyun | sql_calistir | Forza Horizon 4, Bannerlord, Dying Light, Battlefield 1, Pro Soccer Online |
| S6 | İnternetten arkadaşlarla oynanan oyun sayısı | sql_calistir | 'Online Co-op' = 3.895 |

- S3 tuzağı: Placid Plastic Duck Simulator şaka 'Horror' etiketi almış.
- S4 tuzağı: `fiyat` indirimli, doğrusu `normal_fiyat`.
- S6 tuzağı: kullanıcının sözü ("internetten arkadaşlarla") doğru kategoriye çevrilmeli, 'Co-op' (yerel dahil) yanlış.

In [3]:
# 08 · Soru seti 2/6: Yorum / RAG soruları + referans (aracın kendi çıktısı)
from src.rag_araci import yorum_araci

def ekle(kimlik, kategori, soru, beklenen_arac, kontrol, ref_sql=None, ref_arac=None):
    SORULAR.append({"id": kimlik, "kategori": kategori, "soru": soru, "beklenen_arac": beklenen_arac,
                    "kontrol": kontrol, "ref_sql": ref_sql, "ref_arac": ref_arac})

SORULAR = [s for s in SORULAR if s["kategori"] != "rag"]

ekle("R1", "rag", "Baldur's Gate 3 hakkında oyuncular ne düşünüyor?",
     ["oyun_bul", "yorum_ara"], "puan oyun_bul'dan (%96,8); ozetle modu; artı ve eksiler referans yorumlarda geçmeli",
     ref_arac={"mod": "ozetle", "app_id": 1086940})

ekle("R2", "rag", "Elden Ring'de oyuncular en çok neyi eleştiriyor?",
     ["oyun_bul", "yorum_ara"], "olumsuz yorumlara bakmalı (olumlu=0 ya da ozetle); eleştiriler referansta geçmeli",
     ref_arac={"mod": "konu", "soru": "en çok eleştirilen sorunlar", "app_id": 1245620, "olumlu": 0})

ekle("R3", "rag", "Dying Light'ta geceler gerçekten korkutucu mu?",
     ["oyun_bul", "yorum_ara"], "konu modu; ZAYIF EŞLEŞME ve ironik yorumlar kanıt sayılmamalı",
     ref_arac={"mod": "konu", "soru": "geceler korkutucu mu", "app_id": 239140})

ekle("R4", "rag", "Cyberpunk 2077'deki bug'lar zamanla azaldı mı?",
     ["oyun_bul", "yorum_ara"], "hepsi modu; yıllara göre karşılaştırma; 'örneklemdeki N yorum' diye anlatmalı",
     ref_arac={"mod": "hepsi", "soru": "oyunda çok bug var mı", "app_id": 1091500})

ekle("R5", "rag", "Türk oyuncular Mount & Blade II: Bannerlord hakkında ne diyor?",
     ["oyun_bul", "yorum_ara"], "dil='tr' filtresi kullanılmalı; Türkçe yorumlara dayanmalı",
     ref_arac={"mod": "ozetle", "app_id": 261550, "dil": "tr"})

ekle("R6", "rag", "Stray kaç saatte bitiyor, oyuncular ne diyor?",
     ["oyun_bul", "yorum_ara"], "süre yorumlardan (ya da SQL yorumcu_medyan_saat); yorumda olmayan saat uydurmamalı",
     ref_arac={"mod": "konu", "soru": "oyun kaç saatte bitiyor, uzunluğu", "app_id": 1332010})

for s in SORULAR:
    if s["kategori"] != "rag":
        continue
    print(f"\n{'=' * 100}\n[{s['id']}] {s['soru']}\n  beklenen araç: {s['beklenen_arac']} · kontrol: {s['kontrol']}")
    print(f"  referans: yorum_araci({s['ref_arac']})\n")
    print(yorum_araci(**s["ref_arac"]))

print(f"\nToplam soru: {len(SORULAR)}")


[R1] Baldur's Gate 3 hakkında oyuncular ne düşünüyor?
  beklenen araç: ['oyun_bul', 'yorum_ara'] · kontrol: puan oyun_bul'dan (%96,8); ozetle modu; artı ve eksiler referans yorumlarda geçmeli
  referans: yorum_araci({'mod': 'ozetle', 'app_id': 1086940})

ÖZET: örneklemde 200 yorum var (olumlu 194, olumsuz 6). Bu oran genel puan değildir, genel puanı SQL'den al. Aşağıdakiler oyunu değerlendiren en faydalı yorumlar.

=== OLUMLU ===
[app_id=1086940 | olumlu | dil=en | 2023-11-30 | faydalı=8300]
➤ No microtransactions➤ No DRM➤ TONS of content (100+ hours without lazy and repetitive content)➤ TONS of choices, consequences and different ways to play➤ Amazing writing➤ Amazing attention to details➤ Amazing voice acting➤ Best written companions, in-depth characters➤ Beautiful graphics, great animations and facial animations➤ Good NPCs design➤ Single-Player! (optional co-op)➤ Devs listening to feedback and releasing quick updates➤ No CLOWNS in community!Game of the Year!PLEASE Larian never chan

## Soru seti 2/6 · Yorum / RAG (R1–R6)
Referans = aracın kendi çıktısı (`yorum_araci`). Agent'ın her iddiası bu yorumlarda geçmeli, yoksa uydurma.

| id | Soru | Mod | Beklenen cevap |
|---|---|---|---|
| R1 | BG3 hakkında ne düşünüyorlar | ozetle | içerik/seçim/yazım övgüsü; UI yazısı, kamera, görev bug'ı eleştirisi; puan %96,8 oyun_bul'dan |
| R2 | Elden Ring eleştiriler | konu, olumlu=0 | PC portu, stutter, 16:9, anti-cheat, DLC boss tasarımı |
| R3 | Dying Light geceler korkutucu mu | konu | evet, ama güçlü kanıt 1–2 yorum |
| R4 | Cyberpunk bug'lar azaldı mı | hepsi | 2020: 8/1 → 2024: 0/1, düşüş var ama örneklem küçük (19/200) |
| R5 | Türk oyuncular Bannerlord | ozetle, dil=tr | 23 Türkçe yorum: heyecan, Türkçe dil, kasma, ilerleme anlamsız; çoğu 2020 erken erişim |
| R6 | Stray kaç saatte biter | konu | yorumlarda 3–10 saat, çoğu 4–6 |

Bulgular:
- ZAYIF eşiği (0,45) bazen ilgili yorumları da işaretliyor (R2 0,458 stutter yorumu, R3 Çince gece yorumu). Kısa/farklı dilli metinlerde mesafe büyüyor.
- hepsi modu soru cümlesine duyarlı: "bug çok mu?" 37 aday (06), "oyunda çok bug var mı" 19 aday. Agent'ın yazdığı soru sonucu değiştirebilir.

In [4]:
# 08 · Soru seti 3/6: SQL + RAG birlikte + referans (SQL sorgusu ve araç çıktısı)
SORULAR = [s for s in SORULAR if s["kategori"] != "karma"]

ekle("K1", "karma", "The Sims 4'ü oyuncular seviyor mu?",
     ["oyun_bul", "yorum_ara"],
     "puan %87 civarı SQL/oyun_bul'dan; yorumlardaki DLC şikâyetiyle çelişkiyi açıklamalı; korsan/seks modu aktarılmamalı",
     ref_sql="SELECT app_id, ad, olumlu_oran, oy_sayisi, yorum_sayisi, yorum_kapsami, olumlu_csv, olumsuz_csv FROM oyunlar WHERE app_id = 1222670",
     ref_arac={"mod": "ozetle", "app_id": 1222670})

ekle("K2", "karma", "Forza Horizon 4'te Türkçe dil desteği var mı, oyuncular çeviriden memnun mu?",
     ["oyun_bul", "sql_calistir", "yorum_ara"],
     "oyun_dil'den 'Turkish' beyanı + yorumlarla doğrulama (talimat kural 9); 'geliştirici belirtmiş' demeli",
     ref_sql="SELECT o.ad, o.dil_spam, d.dil, d.ses_var FROM oyunlar o JOIN oyun_dil d ON d.app_id = o.app_id WHERE o.app_id = 1293830 AND d.dil = 'Turkish'",
     ref_arac={"mod": "konu", "soru": "Türkçe dil desteği, çeviri kalitesi", "app_id": 1293830})

ekle("K3", "karma", "Battlefield 1'in puanı nasıl, olumsuz yorumlarda ne şikâyet ediliyor?",
     ["oyun_bul", "yorum_ara"],
     "puan oyun_bul'dan; şikâyetler olumsuz yorumlardan (olumlu=0 ya da ozetle), referansta geçmeli",
     ref_sql="SELECT app_id, ad, olumlu_oran, oy_sayisi FROM oyunlar WHERE app_id = 1238840",
     ref_arac={"mod": "konu", "soru": "şikâyet edilen sorunlar", "app_id": 1238840, "olumlu": 0})

ekle("K4", "karma", "Cyberpunk 2077'nin olumlu yorum oranı yıllara göre nasıl değişti, oyuncular ne diyor?",
     ["oyun_bul", "sql_calistir", "yorum_ara"],
     "yıllık oran yorumlar tablosundan GROUP BY ile (bu yorumların oranı, genel puan değil); genel puan ayrıca olumlu_oran'dan; yorumlarla açıklama",
     ref_sql="""SELECT substr(tarih, 1, 4) AS yil, COUNT(*) AS yorum, ROUND(AVG(olumlu), 3) AS olumlu_orani
                FROM yorumlar WHERE app_id = 1091500 GROUP BY yil ORDER BY yil""",
     ref_arac={"mod": "ozetle", "app_id": 1091500})

ekle("K5", "karma", "Half-Life: Alyx VR gözlüğü olmadan oynanabilir mi?",
     ["oyun_bul", "sql_calistir"],
     "oyun_kategori'de 'VR Only' → hayır; yorumlarla destek olursa artı; uydurma mod/çözüm önermemeli",
     ref_sql="SELECT kategori FROM oyun_kategori WHERE app_id = 546560 ORDER BY kategori",
     ref_arac={"mod": "konu", "soru": "VR gözlüğü olmadan oynanabilir mi", "app_id": 546560})

for s in SORULAR:
    if s["kategori"] != "karma":
        continue
    print(f"\n{'=' * 100}\n[{s['id']}] {s['soru']}\n  beklenen araç: {s['beklenen_arac']} · kontrol: {s['kontrol']}")
    print("\n-- SQL referansı --")
    display(referans(s["ref_sql"]))
    print(f"-- Araç referansı: yorum_araci({s['ref_arac']}) --\n")
    print(yorum_araci(**s["ref_arac"]))

print(f"\nToplam soru: {len(SORULAR)}")


[K1] The Sims 4'ü oyuncular seviyor mu?
  beklenen araç: ['oyun_bul', 'yorum_ara'] · kontrol: puan %87 civarı SQL/oyun_bul'dan; yorumlardaki DLC şikâyetiyle çelişkiyi açıklamalı; korsan/seks modu aktarılmamalı

-- SQL referansı --


,app_id,ad,olumlu_oran,oy_sayisi,yorum_sayisi,yorum_kapsami,olumlu_csv,olumsuz_csv
0,1222670,The Sims™ 4,0.874,153114,9140,0.06,5122,4018


-- Araç referansı: yorum_araci({'mod': 'ozetle', 'app_id': 1222670}) --

ÖZET: örneklemde 200 yorum var (olumlu 175, olumsuz 25). Bu oran genel puan değildir, genel puanı SQL'den al. Aşağıdakiler oyunu değerlendiren en faydalı yorumlar.

=== OLUMLU ===
[app_id=1222670 | olumlu | dil=en | 2024-06-30 | faydalı=1488]
best game ever. if you want all DLCs just search up Anadius and enjoy for free.

[app_id=1222670 | olumlu | dil=en | 2022-10-09 | faydalı=1321]
Only People like Elon Musk can buy the whole game! my advise to you is Just to buy the base game and install mods instead.I used mods (Wicked Williams) to turn it into the ULTIMATE SEX game!

[app_id=1222670 | olumlu | dil=zh | 2020-12-19 | faydalı=1181]
我对这个游戏真的是又爱又恨，这游戏又让人入迷，但是定价却让人望而止步，除了新的几个dlc我大部分都买了，我还在在打折的情况下花了一千多（就离谱 )，但不得不说游戏是真好玩，真的把现实生活体现出来了。后面就说一下我个人觉得必买的dlc玩转大学（可以去上大学，然后上课，挺有意思的，特别是把小孩子养大送去大学很有感觉）来去上班（上班996，有很多职业可以做的，推荐游戏主播不错，很赚钱，还有像医生、警察的职业可以亲自去上班，然后根据工作升职，还可以自己创业开店）星梦起飞（当明星，我之前到了五星，出个门一堆人围着我拍照要签名）猫狗总动员（养宠物，各种猫和狗）都会生活（有个城

,ad,dil_spam,dil,ses_var
0,Forza Horizon 4,0,Turkish,1


-- Araç referansı: yorum_araci({'mod': 'konu', 'soru': 'Türkçe dil desteği, çeviri kalitesi', 'app_id': 1293830}) --

KONU ARAMASI: 5 yorum, en ilgiliden aşağı. Mesafe 0.45 üstü = zayıf eşleşme, kanıt sayma.

[app_id=1293830 | olumlu | dil=tr | 2021-03-09 | faydalı=2486 | mesafe=0.332]
Bu muhteşem oyuna türkçe dil ve bölgesel fiyatlandırma getirdiğiniz için türk oyuncular olarak çok teşekkür ederiz.

[app_id=1293830 | olumlu | dil=tr | 2021-03-09 | faydalı=2059 | mesafe=0.342]
Güzel fiyat, Türkçe dil desteği ve on numara yarış oyunu daha ne isteyelim Xbox Türkiye. Playstation gibi para emici olmadığınız için tüm Türk halkı adına teşekkürler.

[app_id=1293830 | olumlu | dil=tr | 2022-09-06 | faydalı=0 | mesafe=0.453 ZAYIF EŞLEŞME]
İndirimdeyken alırsanız içiniz rahat eder. Oynanış efsane grafikler süper (en düşük ayarlarda oynamama rağmen tatmin ediyor. Mağlum şartlarda 6 7 yıllık laptobumda oynuyorum)

[app_id=1293830 | olumlu | dil=tr | 2021-06-14 | faydalı=726 | mesafe=0.484 ZAYIF EŞ

,app_id,ad,olumlu_oran,oy_sayisi
0,1238840,Battlefield™ 1,0.863,163755


-- Araç referansı: yorum_araci({'mod': 'konu', 'soru': 'şikâyet edilen sorunlar', 'app_id': 1238840, 'olumlu': 0}) --

KONU ARAMASI: 5 yorum, en ilgiliden aşağı. Mesafe 0.45 üstü = zayıf eşleşme, kanıt sayma.

[app_id=1238840 | olumsuz | dil=zh | 2022-12-08 | faydalı=2 | mesafe=0.414]
挂巨多，游戏体验特差，谁买谁傻*，解锁枪械要求也巨傻*

[app_id=1238840 | olumsuz | dil=en | 2021-07-06 | faydalı=656 | mesafe=0.457 ZAYIF EŞLEŞME]
This is a great game. Unfortunately, I can not recommend that anyone purchase this game, or BF4, or BFV, or 2042 when it comes out either. And it is not for an emotional reason like "EA bad."The fact is that all of these games are plagued with cheaters, and DICE/EA have COMPLETELY stopped reading reports against players for hacking or harassment, or any player reports for any game other than BFV. because they want you to get bored of the old game and buy the new game, they will let their old games become wastelands filled with cheaters and racists, and the average players who are just t

,yil,yorum,olumlu_orani
0,2020,10646,0.724
1,2021,6209,0.617
2,2022,3878,0.806
3,2023,3597,0.849
4,2024,1960,0.938


-- Araç referansı: yorum_araci({'mod': 'ozetle', 'app_id': 1091500}) --

ÖZET: örneklemde 200 yorum var (olumlu 166, olumsuz 34). Bu oran genel puan değildir, genel puanı SQL'den al. Aşağıdakiler oyunu değerlendiren en faydalı yorumlar.

=== OLUMLU ===
[app_id=1091500 | olumlu | dil=en | 2020-12-09 | faydalı=20114]
I crashed like twice but didn't have a problem after I got the new game ready drivers. I'm literally making this review as a reminder. Had a blast so far but jesus guys, update your nvidia drivers.ULTRA EDIT:I might as well actually review this game in its current state considering the attention this got, while I have not finished it I feel I have experienced enough of the core mechanics to formulate an accurate opinion at least on the gameplay. It is a Witcher 3 reskin with a cyberpunk setting, considering that Witcher 3 was amazing (and I still have yet to get through it entirely) it does translate. Narratively this game excels, the dialogue is well written and acted and t

,kategori
0,Captions available
1,Family Sharing
2,Includes level editor
3,Single-player
4,Steam Achievements
5,Steam Workshop
6,Tracked Controller Support
7,VR Only
8,VR Supported


-- Araç referansı: yorum_araci({'mod': 'konu', 'soru': 'VR gözlüğü olmadan oynanabilir mi', 'app_id': 546560}) --

KONU ARAMASI: 5 yorum, en ilgiliden aşağı. Mesafe 0.45 üstü = zayıf eşleşme, kanıt sayma.

[app_id=546560 | olumlu | dil=zh | 2020-04-21 | faydalı=371 | mesafe=0.279]
如果你没有vr玩不了这款游戏，那能理解。如果你有vr不玩这款游戏，♥♥♥♥♥♥

[app_id=546560 | olumlu | dil=en | 2021-03-21 | faydalı=566 | mesafe=0.325]
If you have a vr headset and dont have this game, you are doing VR all wrong.

[app_id=546560 | olumlu | dil=en | 2020-06-10 | faydalı=1440 | mesafe=0.344]
Only VR game I've played that feels like an actual game and not an extended tech demo.

[app_id=546560 | olumlu | dil=en | 2020-03-24 | faydalı=0 | mesafe=0.345]
By far the best VR game I've ever played, and quite possibly the best game I have played in the past 5 years. If you have VR, buy it. If you don't, watch a walkthrough.

[app_id=546560 | olumlu | dil=en | 2020-04-06 | faydalı=615 | mesafe=0.358]
By far the best VR game out there!  I

## Soru seti 3/6 · SQL + RAG (K1–K5)
| id | Soru | SQL referansı | Yorum referansı / beklenen |
|---|---|---|---|
| K1 | Sims 4 seviliyor mu | %87,4 · 153.114 oy · kapsam 0,06 (CSV oranı %56, puan değil) | DLC fiyatı öfkesi; ilk 2 olumlu yorum korsan (Anadius) + seks modu → aktarılmamalı |
| K2 | FH4 Türkçe + çeviri memnuniyeti | Turkish beyan, dil_spam 0, ses_var 1 (şüpheli) | 2 güçlü teyit; çeviri kalitesi yorumu YOK → kalite uydurmamalı |
| K3 | BF1 puan + şikâyet | %86,3 · 163.755 oy | hile/hacker, rapor sistemi; 5 yorumun 4'ü ZAYIF |
| K4 | Cyberpunk yıllık oran | 2020 %72 · 2021 %62 · 2022 %81 · 2023 %85 · 2024 %94 | ozetle yorumları 2020–21 → tarih belirtmeli |
| K5 | HL Alyx VR'sız | 'VR Only' (+ 'VR Supported', veri çelişkisi) | "VR yoksa oynayamazsın" → hayır |

Bulgular:
- K1: ozetle'nin en faydalı iki olumlu yorumu kural 5 ihlali içeriyor (faydalı oy ≠ ciddiyet).
- K3: ZAYIF eşiği ana şikâyeti (hile) gizleyebilir → kural 3 ile çatışma riski.
- K4: yorumlardan hesaplanan oran genel puan değil ama yıllık eğilim için tek kaynak.

In [5]:
# 08 · Soru seti 4/6: Öneri soruları + referans (kurallara göre SQL / oyun_ara çıktısı)
from src.rag_araci import oyun_araci

SORULAR = [s for s in SORULAR if s["kategori"] != "oneri"]

COCUK_HARIC = "'Horror','Psychological Horror','Survival Horror','Gore','Violent','Blood','Sexual Content','Nudity','Mature','NSFW'"

ekle("O1", "oneri", "Bana bir korku oyunu önerir misin?",
     ["sql_calistir"], "Horror etiketi + oran ≥ 0,85 + oy sayısına göre; yetiskin=0, iskelet=0; çocuk hariç listesi UYGULANMAMALI",
     ref_sql="""SELECT o.ad, o.olumlu_oran, o.oy_sayisi FROM oyunlar o
                WHERE o.olumlu_oran >= 0.85 AND o.oy_sayisi >= 100 AND o.yetiskin = 0 AND o.iskelet = 0
                  AND EXISTS (SELECT 1 FROM oyun_etiket e WHERE e.app_id = o.app_id AND e.etiket = 'Horror')
                ORDER BY o.oy_sayisi DESC LIMIT 10""")

ekle("O2", "oneri", "8 yaşındaki çocuğuma hangi oyunu alabilirim?",
     ["sql_calistir"], "Family Friendly + hariç etiketler NOT EXISTS + yetiskin=0; korku oyunu (RE Village) OLMAMALI; sonda ebeveyn uyarısı",
     ref_sql=f"""SELECT o.ad, o.olumlu_oran, o.oy_sayisi FROM oyunlar o
                WHERE o.olumlu_oran >= 0.85 AND o.oy_sayisi >= 100 AND o.yetiskin = 0 AND o.iskelet = 0
                  AND EXISTS (SELECT 1 FROM oyun_etiket e WHERE e.app_id = o.app_id AND e.etiket = 'Family Friendly')
                  AND NOT EXISTS (SELECT 1 FROM oyun_etiket e WHERE e.app_id = o.app_id AND e.etiket IN ({COCUK_HARIC}))
                ORDER BY o.oy_sayisi DESC LIMIT 10""")

ekle("O3", "oneri", "Uzayda geçen bir keşif oyunu önerir misin?",
     ["oyun_ara", "sql_calistir"], "oyun_ara ile aday + SQL ile puan kontrolü; listede olmayan oyun uydurmamalı",
     ref_arac={"istek": "uzayda geçen keşif oyunu"})

ekle("O4", "oneri", "Arkadaşlarımla internetten birlikte oynayabileceğim bir korku oyunu öner.",
     ["sql_calistir"], "kesin kriter → SQL: 'Online Co-op' kategori + 'Horror' etiket, öneri sıralaması",
     ref_sql="""SELECT o.ad, o.olumlu_oran, o.oy_sayisi FROM oyunlar o
                WHERE o.olumlu_oran >= 0.85 AND o.oy_sayisi >= 100 AND o.yetiskin = 0 AND o.iskelet = 0
                  AND EXISTS (SELECT 1 FROM oyun_etiket e WHERE e.app_id = o.app_id AND e.etiket = 'Horror')
                  AND EXISTS (SELECT 1 FROM oyun_kategori k WHERE k.app_id = o.app_id AND k.kategori = 'Online Co-op')
                ORDER BY o.oy_sayisi DESC LIMIT 10""")

ekle("O5", "oneri", "Türkçe destekli, rahatlatıcı bir çiftlik oyunu önerir misin?",
     ["oyun_ara", "sql_calistir"], "tarif (oyun_ara) + Türkçe kesin kriter (SQL oyun_dil); Türkçesi olmayan oyun önermemeli",
     ref_sql="""SELECT o.ad, o.olumlu_oran, o.oy_sayisi, o.dil_spam FROM oyunlar o
                WHERE o.olumlu_oran >= 0.85 AND o.oy_sayisi >= 100 AND o.yetiskin = 0 AND o.iskelet = 0
                  AND EXISTS (SELECT 1 FROM oyun_etiket e WHERE e.app_id = o.app_id AND e.etiket = 'Farming Sim')
                  AND EXISTS (SELECT 1 FROM oyun_dil d WHERE d.app_id = o.app_id AND d.dil = 'Turkish')
                ORDER BY o.oy_sayisi DESC LIMIT 10""",
     ref_arac={"istek": "rahatlatıcı çiftlik oyunu"})

for s in SORULAR:
    if s["kategori"] != "oneri":
        continue
    print(f"\n{'=' * 100}\n[{s['id']}] {s['soru']}\n  beklenen araç: {s['beklenen_arac']} · kontrol: {s['kontrol']}")
    if s["ref_sql"]:
        print("\n-- SQL referansı --")
        display(referans(s["ref_sql"]))
    if s["ref_arac"]:
        print(f"-- oyun_ara referansı: oyun_araci({s['ref_arac']}) --\n")
        print(oyun_araci(**s["ref_arac"]))

print("\n-- RE Village kontrolü (O1'de olabilir, O2'de olmamalı) --")
display(referans("""SELECT app_id, ad, olumlu_oran, oy_sayisi, yetiskin, yetiskin_etiket_sayisi FROM oyunlar
                    WHERE ad LIKE 'Resident Evil Village%'"""))
print(f"Toplam soru: {len(SORULAR)}")


[O1] Bana bir korku oyunu önerir misin?
  beklenen araç: ['sql_calistir'] · kontrol: Horror etiketi + oran ≥ 0,85 + oy sayısına göre; yetiskin=0, iskelet=0; çocuk hariç listesi UYGULANMAMALI

-- SQL referansı --


,ad,olumlu_oran,oy_sayisi
0,Phasmophobia,0.959,719060
1,Lethal Company,0.973,443317
2,Dying Light,0.952,435283
3,7 Days to Die,0.882,342380
4,Deep Rock Galactic,0.972,314871
5,Sons Of The Forest,0.865,209485
6,DOOM Eternal,0.911,196319
7,Ready or Not,0.880,195584
8,Resident Evil 4,0.976,154523
9,Stray,0.973,146316



[O2] 8 yaşındaki çocuğuma hangi oyunu alabilirim?
  beklenen araç: ['sql_calistir'] · kontrol: Family Friendly + hariç etiketler NOT EXISTS + yetiskin=0; korku oyunu (RE Village) OLMAMALI; sonda ebeveyn uyarısı

-- SQL referansı --


,ad,olumlu_oran,oy_sayisi
0,Hogwarts Legacy,0.907,293388
1,The Sims™ 4,0.874,153114
2,Totally Accurate Battle Simulator,0.978,129125
3,Stumble Guys,0.851,121175
4,Farming Simulator 22,0.922,79161
5,Golf With Your Friends,0.883,64892
6,PowerWash Simulator,0.968,48209
7,The Henry Stickmin Collection,0.987,46994
8,Brick Rigs,0.943,45109
9,LEGO® Star Wars™: The Skywalker Saga,0.910,43672



[O3] Uzayda geçen bir keşif oyunu önerir misin?
  beklenen araç: ['oyun_ara', 'sql_calistir'] · kontrol: oyun_ara ile aday + SQL ile puan kontrolü; listede olmayan oyun uydurmamalı
-- oyun_ara referansı: oyun_araci({'istek': 'uzayda geçen keşif oyunu'}) --

OYUN ARAMASI (tarife göre benzerlik + popülerlik, en uygundan aşağı). Puan/fiyat/etiket gibi kesin bilgileri SQL'den kontrol et.
app_id,ad,oy_sayisi,mesafe
1716740,Starfield,160947,0.382
973810,Journey To The Savage Planet,5792,0.354
1128920,EVERSPACE™ 2,10841,0.37
753640,Outer Wilds,73237,0.403
1750770,Starcom: Unknown Space,2005,0.365
2925510,Voyager-19,584,0.371
1194050,Orange Cast: Sci-Fi Space Action Game,158,0.357
954850,Kerbal Space Program 2,27050,0.432
1356480,helionaut,201,0.372
383120,Empyrion - Galactic Survival,32621,0.44


[O4] Arkadaşlarımla internetten birlikte oynayabileceğim bir korku oyunu öner.
  beklenen araç: ['sql_calistir'] · kontrol: kesin kriter → SQL: 'Online Co-op' kategori + 'Horror' etiket, öneri sıral

,ad,olumlu_oran,oy_sayisi
0,Phasmophobia,0.959,719060
1,Lethal Company,0.973,443317
2,Dying Light,0.952,435283
3,7 Days to Die,0.882,342380
4,Deep Rock Galactic,0.972,314871
5,Sons Of The Forest,0.865,209485
6,Ready or Not,0.880,195584
7,Content Warning,0.944,105937
8,Escape the Backrooms,0.910,87214
9,DEVOUR,0.911,79359



[O5] Türkçe destekli, rahatlatıcı bir çiftlik oyunu önerir misin?
  beklenen araç: ['oyun_ara', 'sql_calistir'] · kontrol: tarif (oyun_ara) + Türkçe kesin kriter (SQL oyun_dil); Türkçesi olmayan oyun önermemeli

-- SQL referansı --


,ad,olumlu_oran,oy_sayisi,dil_spam
0,Farming Simulator 22,0.922,79161,0
1,Potion Craft: Alchemist Simulator,0.913,32097,0
2,Timberborn,0.952,27568,0
3,Travellers Rest,0.890,11466,0
4,Ravenous Devils,0.926,8254,0
5,Atomicrops,0.923,5274,0
6,Rival Stars Horse Racing: Desktop Edition,0.926,3871,0
7,Outpath,0.950,2374,0
8,Nova Lands,0.917,2351,0
9,Farm Together 2,0.900,2329,0


-- oyun_ara referansı: oyun_araci({'istek': 'rahatlatıcı çiftlik oyunu'}) --

OYUN ARAMASI (tarife göre benzerlik + popülerlik, en uygundan aşağı). Puan/fiyat/etiket gibi kesin bilgileri SQL'den kontrol et.
app_id,ad,oy_sayisi,mesafe
1137750,Farmer's Life,1995,0.331
1119730,"Ranch Simulator: Build, Hunt, Farm",29154,0.38
1350840,Song Of The Prairie,1258,0.341
2111170,STORY OF SEASONS: A Wonderful Life,1125,0.346
1123830,Farm Manager 2021,2085,0.36
978780,STORY OF SEASONS: Friends of Mineral Town,5051,0.373
1248130,Farming Simulator 22,79161,0.412
2418520,Farm Together 2,2329,0.368
1403650,Everdream Valley,739,0.356
1231520,Farmtale,263,0.348


-- RE Village kontrolü (O1'de olabilir, O2'de olmamalı) --


,app_id,ad,olumlu_oran,oy_sayisi,yetiskin,yetiskin_etiket_sayisi
0,1196590,Resident Evil Village,0.956,113043,0,1


Toplam soru: 22


## Soru seti 4/6 · Öneri (O1–O5)
Referans = 07 kurallarıyla yazılmış SQL (öner: oran ≥ 0,85, oy sayısına göre) + oyun_ara çıktısı.

| id | Soru | Referans (ilk sıralar) | Kritik kontrol |
|---|---|---|---|
| O1 | Korku öner | Phasmophobia, Lethal Company, Dying Light, 7DtD, RE4 | çocuk hariç listesi uygulanmamalı |
| O2 | 8 yaş çocuk | Hogwarts Legacy, Sims 4, TABS, Stumble Guys, FS22 | korku yok + ebeveyn uyarısı (etiketler yaş sınırını tam yakalamıyor) |
| O3 | Uzay keşif | Starfield, Outer Wilds, Everspace 2, Journey to the Savage Planet | SQL kontrolü → Starfield %57 uyarısı |
| O4 | Online co-op korku | Phasmophobia, Lethal Company, Content Warning, DEVOUR | SQL ile iki kesin kriter |
| O5 | Türkçe rahatlatıcı çiftlik | kesişim: Farming Simulator 22, Farm Together 2 | Türkçesi doğrulanmamış oyun önermemeli |

- RE Village: yetiskin = 0 (1 yetişkin etiketi), 113.043 oy → O1 sıralamasında 11.
- Bulgu: etiket filtresi etiketin sırasına bakmıyor → Deep Rock Galactic, DOOM Eternal, Stray "Horror"; Ravenous Devils "Farming Sim". Olası iyileştirme: etiketin ilk N sırada olması şartı (ölçümden sonra).

In [6]:
# 08 · Soru seti 5/6: Tuzak soruları + referans
from src.sql_araci import oyun_bul_araci

SORULAR = [s for s in SORULAR if s["kategori"] != "tuzak"]

ekle("T1", "tuzak", "Starfield hakkında oyuncular ne diyor?",
     ["oyun_bul", "yorum_ara"], "yorum verisi yok demeli; puan %57 oyun_bul'dan; yorum UYDURMAMALI",
     ref_sql="SELECT app_id, ad, olumlu_oran, oy_sayisi, yorum_var, chroma_yorum_sayisi FROM oyunlar WHERE app_id = 1716740",
     ref_arac={"mod": "ozetle", "app_id": 1716740})

ekle("T2", "tuzak", "The Witcher 3'ün puanı kaç?",
     ["oyun_bul"], "veri setinde yok demeli; benzer adlı başka oyunun puanını vermemeli; eğitimden bildiği puanı söylememeli",
     ref_sql="SELECT app_id, ad, olumlu_oran, oy_sayisi FROM oyunlar WHERE ad LIKE '%Witcher%' ORDER BY oy_sayisi DESC",
     ref_arac={"arac": "oyun_bul", "isim": "The Witcher 3"})

ekle("T3", "tuzak", "Spider-Man oyununun puanı kaç?",
     ["oyun_bul"], "birden fazla aday: hangisini seçtiğini söylemeli ve diğerini adıyla anmalı (ya da sormalı)",
     ref_arac={"arac": "oyun_bul", "isim": "Spider-Man"})

aday = referans("""SELECT app_id, ad, olumlu_oran, oy_sayisi, chroma_yorum_sayisi FROM oyunlar
                   WHERE oy_sayisi BETWEEN 20 AND 60 AND chroma_yorum_sayisi BETWEEN 5 AND 20
                     AND iskelet = 0 AND yetiskin = 0 AND dil_spam = 0
                   ORDER BY oy_sayisi DESC, app_id LIMIT 1""").iloc[0]
ekle("T4", "tuzak", f"{aday['ad']} nasıl bir oyun, alınır mı?",
     ["oyun_bul", "yorum_ara"], "oy < 100 → 'az oy var, puan güvenilir değil' uyarısı (kural 5); yorum sayısı az olduğunu söylemeli",
     ref_sql=f"SELECT app_id, ad, olumlu_oran, oy_sayisi, yorum_sayisi, chroma_yorum_sayisi FROM oyunlar WHERE app_id = {aday['app_id']}",
     ref_arac={"mod": "ozetle", "app_id": int(aday["app_id"])})

ekle("T5", "tuzak", "En çok oy alan soulslike oyunlar hangileri?",
     ["sql_calistir"], "etiket yazılışı 'Souls-like'; 0 satır dönerse 'yok' demeden yazılışı kontrol etmeli (SQL kural 8–9)",
     ref_sql="""SELECT o.ad, o.olumlu_oran, o.oy_sayisi FROM oyunlar o
                WHERE o.yetiskin = 0 AND o.iskelet = 0
                  AND EXISTS (SELECT 1 FROM oyun_etiket e WHERE e.app_id = o.app_id AND e.etiket = 'Souls-like')
                ORDER BY o.oy_sayisi DESC LIMIT 10""")

def arac_referansi(ref):
    if ref.get("arac") == "oyun_bul":
        return oyun_bul_araci(ref["isim"])
    if "istek" in ref:
        return oyun_araci(**ref)
    return yorum_araci(**ref)

for s in SORULAR:
    if s["kategori"] != "tuzak":
        continue
    print(f"\n{'=' * 100}\n[{s['id']}] {s['soru']}\n  beklenen araç: {s['beklenen_arac']} · kontrol: {s['kontrol']}")
    if s["ref_sql"]:
        print("\n-- SQL referansı --")
        display(referans(s["ref_sql"]))
    if s["ref_arac"]:
        print(f"-- Araç referansı: {s['ref_arac']} --\n")
        print(arac_referansi(s["ref_arac"]))

print("\n-- T5 yazılış kontrolü --")
display(referans("SELECT DISTINCT etiket FROM oyun_etiket WHERE etiket LIKE '%soul%'"))
print(f"Toplam soru: {len(SORULAR)}")


[T1] Starfield hakkında oyuncular ne diyor?
  beklenen araç: ['oyun_bul', 'yorum_ara'] · kontrol: yorum verisi yok demeli; puan %57 oyun_bul'dan; yorum UYDURMAMALI

-- SQL referansı --


,app_id,ad,olumlu_oran,oy_sayisi,yorum_var,chroma_yorum_sayisi
0,1716740,Starfield,0.57,160947,0,0


-- Araç referansı: {'mod': 'ozetle', 'app_id': 1716740} --

Bu oyun için yorum verisi yok. Sadece SQL'deki sayıları kullan ve bunu kullanıcıya söyle.

[T2] The Witcher 3'ün puanı kaç?
  beklenen araç: ['oyun_bul'] · kontrol: veri setinde yok demeli; benzer adlı başka oyunun puanını vermemeli; eğitimden bildiği puanı söylememeli

-- SQL referansı --


,app_id,ad,olumlu_oran,oy_sayisi
0,1284410,GWENT: The Witcher Card Game,0.821,28743
1,2376280,Whisker Witchery,0.859,64
2,1594560,Chrome Switcher,1.000,2
3,2889630,The Witcher 3 REDkit Playtest,NaN,0


-- Araç referansı: {'arac': 'oyun_bul', 'isim': 'The Witcher 3'} --

'The Witcher 3' için adı en çok benzeyen 5 oyun (benzerlik = ad benzerliği 0-100, kesin eşleşme anlamına gelmez; olumlu_oran = Steam oylarından genel puan, 0-1, boşsa oy yok):
app_id,ad,oy_sayisi,olumlu_oran,benzerlik
2889630,The Witcher 3 REDkit Playtest,0,,90
1291710,The Watcher,0,,86
3170160,The Writ,2,0.5,83
1284410,GWENT: The Witcher Card Game,28743,0.821,82
1112830,The Watchers,468,0.754,82
Kural: Sorulan oyunu ad ve oy sayısına bakarak seç. Adaylardan hiçbiri sorulan oyun değilse (farklı bir oyun, Playtest, araç veya kart oyunu) oyunun veri setinde olmadığını söyle. Seçtiğin oyunu sonraki SQL sorgularında app_id ile kullan.

[T3] Spider-Man oyununun puanı kaç?
  beklenen araç: ['oyun_bul'] · kontrol: birden fazla aday: hangisini seçtiğini söylemeli ve diğerini adıyla anmalı (ya da sormalı)
-- Araç referansı: {'arac': 'oyun_bul', 'isim': 'Spider-Man'} --

'Spider-Man' için adı en çok benzeyen 5 oyun (benzerlik =

,app_id,ad,olumlu_oran,oy_sayisi,yorum_sayisi,chroma_yorum_sayisi
0,2002600,吞食一统中原Tunshi unifying plains,0.946,56,56,16


-- Araç referansı: {'mod': 'ozetle', 'app_id': 2002600} --

ÖZET: örneklemde 16 yorum var (olumlu 13, olumsuz 3). Bu oran genel puan değildir, genel puanı SQL'den al. Aşağıdakiler oyunu değerlendiren en faydalı yorumlar.

=== OLUMLU ===
[app_id=2002600 | olumlu | dil=zh | 2024-08-29 | faydalı=1]
满满的童年回忆，仿佛回到了红白机的那个时候通宵打章节，大爱。

[app_id=2002600 | olumlu | dil=zh | 2024-08-14 | faydalı=0]
是童年的回忆，画面很不错，有人物专属特技，每个武将特色鲜明，好评！！

[app_id=2002600 | olumlu | dil=zh | 2024-08-14 | faydalı=0]
是童年的回忆，画面很不错，有人物专属特技，每个武将特色鲜明，好评！！！愿意推荐此游戏 期待三周目期待通天塔模式

[app_id=2002600 | olumlu | dil=zh | 2024-08-13 | faydalı=0]
是童年的回忆，画面高级，有人物专属特技，每个武将特色鲜明，好评！！！愿意推荐此游戏

[app_id=2002600 | olumlu | dil=zh | 2024-09-29 | faydalı=0]
游戏还行，一周目没问题，二周目剧情问题多多牛头不对马嘴，将就玩下了

[app_id=2002600 | olumlu | dil=zh | 2024-09-26 | faydalı=0]
好玩。比起其他吞食这个玩法很丰富，推荐！

=== OLUMSUZ ===
[app_id=2002600 | olumsuz | dil=zh | 2024-08-15 | faydalı=13]
目前不要买，改键就很麻烦，进游戏加速还只能加速走路，打架跟蜗牛一样，两分钟打一次，完全没有刷刷刷的快感，攻略还要50块钱买，不然就很容易错过关键东西。2024.9.15补充：所谓的64卦完全是坑人的，

,ad,olumlu_oran,oy_sayisi
0,ELDEN RING,0.926,977477
1,Black Myth: Wukong,0.963,958121
2,God of War,0.961,145482
3,Ori and the Will of the Wisps,0.966,131313
4,ARMORED CORE™ VI FIRES OF RUBICON™,0.912,71495
5,OUTRIDERS,0.664,60139
6,STAR WARS Jedi: Survivor™,0.676,56991
7,REMNANT II®,0.834,56125
8,Hades II,0.943,50050
9,Ghost of Tsushima DIRECTOR'S CUT,0.928,45338



-- T5 yazılış kontrolü --


,etiket
0,Souls-like


Toplam soru: 27


## Soru seti 5/6 · Tuzak (T1–T5)
| id | Soru | Referans | Doğru davranış |
|---|---|---|---|
| T1 | Starfield yorumları | %57 · 160.947 oy · yorum_var 0 | "yorum verisi yok" + sadece SQL sayıları |
| T2 | Witcher 3 puanı | adaylar: REDkit Playtest (0 oy, benzerlik 90), GWENT | "veri setinde yok", başka oyunun puanı yok |
| T3 | Spider-Man puanı | Remastered %95,9 · Miles Morales %93,3 (ikisi de benzerlik 90) | seçtiğini söyle + diğerini an / sor |
| T4 | 吞食一统中原 alınır mı | %94,6 · 56 oy · 16 RAG yorumu | "az oy, puan güvenilir değil"; aynı kalıpta tekrar eden olumlu yorumlar |
| T5 | En çok oylanan soulslike | 'Souls-like': Elden Ring, Wukong, God of War, Ori... | yazılışı kontrol et, 0 satırda "yok" deme |

- T4 adayı veriden otomatik seçildi (oy 20–60, RAG'da 5–20 yorum, ORDER BY oy, app_id → her seferinde aynı).
- T5'te de etiket gevşekliği: God of War, Ori, Hades II "Souls-like".

In [8]:
# 08 · Soru seti 6/6: kapsam dışı + hafıza, referans metinlerini dondur, dosyaya kaydet
import json
from pathlib import Path

def ekle(kimlik, kategori, soru, beklenen_arac, kontrol, ref_sql=None, ref_arac=None, onceki=None):
    SORULAR.append({"id": kimlik, "kategori": kategori, "soru": soru, "beklenen_arac": beklenen_arac,
                    "kontrol": kontrol, "ref_sql": ref_sql, "ref_arac": ref_arac, "onceki": onceki})

SORULAR = [s for s in SORULAR if s["kategori"] != "diger"]

ekle("D1", "diger", "Merhaba, nasılsın?",
     [], "araç çağırmamalı; kısa selam + neler yapabildiğini söylemeli")

ekle("D2", "diger", "Python'da bir listeyi nasıl ters çeviririm?",
     [], "araç çağırmamalı; sadece Steam oyunlarında yardımcı olabildiğini kibarca söylemeli, kod yazmamalı")

ekle("D3", "diger", "Peki oyuncular en çok neyi eleştiriyor?",
     ["yorum_ara"], "önceki soru Elden Ring: oyun_bul'u tekrar çağırmadan app_id 1245620 ile yorum_ara; eleştiriler referansta geçmeli",
     ref_arac={"mod": "konu", "soru": "en çok eleştirilen sorunlar", "app_id": 1245620, "olumlu": 0},
     onceki="Elden Ring'in puanı kaç?")

for s in SORULAR:
    for alan in ["ref_sql", "ref_arac", "onceki"]:
        s.setdefault(alan, None)
    parcalar = []
    if s["ref_sql"]:
        parcalar.append("SQL REFERANSI:\n" + referans(s["ref_sql"]).to_csv(index=False))
    if s["ref_arac"]:
        parcalar.append("ARAÇ REFERANSI:\n" + arac_referansi(s["ref_arac"]))
    s["referans_metin"] = "\n\n".join(parcalar)

DOSYA = Path("../reports/08_soru_seti.json")
DOSYA.write_text(json.dumps(SORULAR, ensure_ascii=False, indent=2), encoding="utf-8")

df_sorular = pd.DataFrame(SORULAR)
df_sorular["referans_kar"] = df_sorular["referans_metin"].str.len()
print(df_sorular["kategori"].value_counts().to_string())
print(f"\nToplam: {len(df_sorular)} soru · kaydedildi: {DOSYA} ({DOSYA.stat().st_size / 1024:.1f} KB)\n")
display(df_sorular[["id", "kategori", "soru", "beklenen_arac", "onceki", "referans_kar"]])

kategori
sql      6
rag      6
karma    5
oneri    5
tuzak    5
diger    3

Toplam: 30 soru · kaydedildi: ../reports/08_soru_seti.json (82.5 KB)



,id,kategori,soru,beklenen_arac,onceki,referans_kar
0,S1,sql,Baldur's Gate 3'ün Steam puanı ve oy sayısı nedir?,[oyun_bul],NaN,84
1,S2,sql,100 binden fazla oy almış kaç oyun var?,[sql_calistir],NaN,30
2,S3,sql,Türkçe destekleyen en yüksek puanlı korku oyunları hangileri?,[sql_calistir],NaN,446
3,S4,sql,Cyberpunk 2077 kaç dolar?,"[oyun_bul, sql_calistir]",NaN,96
4,S5,sql,En çok Türkçe yorum alan 5 oyun hangisi?,[sql_calistir],NaN,198
5,S6,sql,İnternetten arkadaşlarla birlikte oynanabilen kaç oyun var?,[sql_calistir],NaN,32
6,R1,rag,Baldur's Gate 3 hakkında oyuncular ne düşünüyor?,"[oyun_bul, yorum_ara]",NaN,7606
7,R2,rag,Elden Ring'de oyuncular en çok neyi eleştiriyor?,"[oyun_bul, yorum_ara]",NaN,3627
8,R3,rag,Dying Light'ta geceler gerçekten korkutucu mu?,"[oyun_bul, yorum_ara]",NaN,3777
9,R4,rag,Cyberpunk 2077'deki bug'lar zamanla azaldı mı?,"[oyun_bul, yorum_ara]",NaN,2612


## Soru seti 6/6 · Kapsam dışı + hafıza (D1–D3) ve kayıt
| id | Soru | Doğru davranış |
|---|---|---|
| D1 | Merhaba, nasılsın? | araçsız, kısa selam + neler yapabildiği |
| D2 | Python'da liste ters çevirme | araçsız, kibar ret, kod yok |
| D3 | (önce "Elden Ring'in puanı kaç?") Peki oyuncular en çok neyi eleştiriyor? | geçmişten app_id, oyun_bul tekrar yok; eleştiriler R2 referansında |

**Soru seti tamam: 30 soru** → `reports/08_soru_seti.json`
| Kategori | Sayı | Ne ölçüyor |
|---|---|---|
| sql | 6 | doğru sütun/tablo, COUNT, kullanıcı dilini kategoriye çevirme |
| rag | 6 | doğru mod, iddiaların yorumlarda geçmesi |
| karma | 5 | sayı SQL'den + "neden" yorumlardan, çelişkiyi açıklama |
| oneri | 5 | kural 6 sıralaması, çocuk filtresi, tarif + kesin kriter |
| tuzak | 5 | "yok / emin değilim / hangisi?" diyebilmek |
| diger | 3 | araçsız cevap, kapsam dışı ret, hafıza |

- Her soruda: `beklenen_arac`, `kontrol` (doğru sayılma şartı), `referans_metin`.
- Referans metinleri donduruldu: hepsi modu gibi değişken çıktılar puanlamayı etkilemesin; ben de LLM hakem de aynı metne bakıyoruz.

In [9]:
# 08 · Değerlendirme döngüsü: 30 soru × 3 tekrar, her çalıştırma ayrı oturum, sonuçlar jsonl'a
import time
from datetime import datetime
from tqdm.auto import tqdm
from src.agent_graf import GRAF

SORULAR = json.loads(Path("../reports/08_soru_seti.json").read_text(encoding="utf-8"))
TEKRAR = 3
MODEL = VARSAYILAN_MODEL
SONUC_DOSYA = Path("../reports/08_calistirmalar.jsonl")
FIYAT = {"gpt-4.1-mini": (0.40, 1.60)}   # $ / 1 milyon token: (girdi, çıktı)

def yapilanlar():
    if not SONUC_DOSYA.exists():
        return set()
    with open(SONUC_DOSYA, encoding="utf-8") as f:
        return {(k["id"], k["tekrar"], k["model"], k["metin_modu"]) for k in map(json.loads, f)}

def arac_ciktilari(oturum):
    mesajlar = GRAF.get_state({"configurable": {"thread_id": oturum}}).values["mesajlar"]
    return [m["content"] for m in mesajlar if m.get("role") == "tool"]

def bir_calistirma(s, tekrar):
    oturum = f"08-{s['id']}-{tekrar}-{datetime.now():%H%M%S}"
    if s["onceki"]:
        calistir(s["onceki"], oturum=oturum, model=MODEL)
    r = calistir(s["soru"], oturum=oturum, model=MODEL)
    gf, cf = FIYAT[MODEL]
    return {"id": s["id"], "kategori": s["kategori"], "tekrar": tekrar, "model": MODEL,
            "metin_modu": ayarlar.METIN_MODU, "talimat_kar": len(SISTEM_TALIMATI),
            "soru": s["soru"], "cevap": r["cevap"],
            "arac_adlari": [a["ad"] for a in r["iz"] if a["tip"] == "arac"], "araclar": r["araclar"],
            "arac_hata": sum(a.get("hata", False) for a in r["iz"]),
            "tur": r["tur"], "sure": r["sure"], "girdi_token": r["girdi_token"], "cikti_token": r["cikti_token"],
            "maliyet_usd": round((r["girdi_token"] * gf + r["cikti_token"] * cf) / 1e6, 5),
            "arac_ciktilari": arac_ciktilari(oturum),
            "zaman": datetime.now().isoformat(timespec="seconds"), "hata": None}

bitti = yapilanlar()
isler = [(s, t) for t in range(1, TEKRAR + 1) for s in SORULAR
         if (s["id"], t, MODEL, ayarlar.METIN_MODU) not in bitti]
print(f"Model: {MODEL} · METIN_MODU: {ayarlar.METIN_MODU} · talimat: {len(SISTEM_TALIMATI)} kar.")
print(f"Yapılacak: {len(isler)} çalıştırma (daha önce biten: {len(bitti)})")

t0 = time.perf_counter()
for s, t in tqdm(isler):
    try:
        kayit = bir_calistirma(s, t)
    except Exception as e:
        kayit = {"id": s["id"], "kategori": s["kategori"], "tekrar": t, "model": MODEL,
                 "metin_modu": ayarlar.METIN_MODU, "soru": s["soru"], "cevap": None,
                 "zaman": datetime.now().isoformat(timespec="seconds"), "hata": f"{type(e).__name__}: {e}"}
    with open(SONUC_DOSYA, "a", encoding="utf-8") as f:
        f.write(json.dumps(kayit, ensure_ascii=False) + "\n")
print(f"Bitti: {(time.perf_counter() - t0) / 60:.1f} dk")

df = pd.read_json(SONUC_DOSYA, lines=True)
df = df[(df["model"] == MODEL) & (df["metin_modu"] == ayarlar.METIN_MODU)]
print(f"\nKayıt: {len(df)} · hatalı çalıştırma: {df['hata'].notna().sum()} · araç hatası içeren: {(df['arac_hata'] > 0).sum()}")
print(f"Toplam maliyet: ${df['maliyet_usd'].sum():.3f} · ortalama süre: {df['sure'].mean():.1f} sn · ortalama tur: {df['tur'].mean():.1f}")
display(df.groupby("kategori")[["tur", "sure", "girdi_token", "cikti_token", "maliyet_usd"]].mean().round(2))
display(df[df["hata"].notna()][["id", "tekrar", "hata"]])

/home/kaan/steam-oyun-asistani/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model: gpt-4.1-mini · METIN_MODU: gizle · talimat: 3702 kar.
Yapılacak: 90 çalıştırma (daha önce biten: 0)


100%|██████████| 90/90 [05:43<00:00,  3.82s/it]

Bitti: 5.7 dk

Kayıt: 90 · hatalı çalıştırma: 0 · araç hatası içeren: 1
Toplam maliyet: $0.462 · ortalama süre: 3.7 sn · ortalama tur: 2.6


,tur,sure,girdi_token,cikti_token,maliyet_usd
kategori,,,,,
diger,1.33,1.58,6415.67,83.00,0.00
karma,3.47,4.61,16823.93,226.47,0.01
oneri,2.20,4.03,9608.33,304.13,0.00
rag,3.11,5.07,15972.00,256.17,0.01
sql,2.33,3.14,10046.89,147.50,0.00
tuzak,2.40,3.02,10564.07,147.93,0.00


,id,tekrar,hata


## Değerlendirme çalıştırması · gpt-4.1-mini · gizle · talimat v4
- 30 soru × 3 tekrar = 90 çalıştırma, her biri ayrı LangGraph oturumunda (D3 hariç: önce Elden Ring sorusu aynı oturumda).
- Sonuçlar: `reports/08_calistirmalar.jsonl` (cevap, araçlar, araç çıktıları, tur, süre, token, maliyet). Kesilirse kaldığı yerden devam eder.
- 5,7 dk · çökme 0 · araç hatası içeren 1 çalıştırma
- Toplam $0,46 → soru başı ≈ 0,5 sent · ortalama 3,7 sn · 2,6 tur

| Kategori | Tur | Süre (sn) | Girdi token |
|---|---|---|---|
| diger | 1,3 | 1,6 | 6.416 |
| sql | 2,3 | 3,1 | 10.047 |
| tuzak | 2,4 | 3,0 | 10.564 |
| oneri | 2,2 | 4,0 | 9.608 |
| rag | 3,1 | 5,1 | 15.972 |
| karma | 3,5 | 4,6 | 16.824 |

- En pahalı: karma ve rag (yorum metinleri bağlamı büyütüyor).
- Maliyete hepsi modunun içindeki etiketleme çağrıları dahil değil.

In [11]:
# 08 · Ön kıyas hazırlığı: gpt-6-luna'yı API'de bul, agent içinde tek soruyla dene
from src import llm

adlar = sorted(m.id for m in llm.istemci("gpt-4.1-mini").models.list() if "luna" in m.id or m.id.startswith("gpt-6"))
print("API'deki gpt-6 / luna modelleri:", adlar)

try:
    s = calistir("Elden Ring'in puanı kaç?", oturum="08-luna-deneme", model="gpt-6-luna")
    print("\nCevap:", s["cevap"])
    print("Araçlar:", s["araclar"])
    print(f'{s["tur"]} tur · {s["sure"]} sn · girdi {s["girdi_token"]:,} / çıktı {s["cikti_token"]:,} token')
except Exception as e:
    print(f"\nHATA: {type(e).__name__}: {e}")

API'deki gpt-6 / luna modelleri: ['gpt-5.6-luna', 'gpt-6-astra', 'gpt-6-luna', 'gpt-6-sol', 'gpt-6.1-sol', 'gpt-6.1-terra']

Cevap: Elden Ring’in Steam oylarına göre olumlu puanı **%92,6**. Bu oran **977.477** oya dayanıyor.
Araçlar: ['oyun_bul({"isim":"Elden Ring"})', 'sql_calistir({"sorgu":"SELECT ad, olumlu_oran, oy_sayisi FROM oyunlar WHERE app_id = 1245620;"})']
3 tur · 4.6 sn · girdi 13,420 / çıktı 100 token


In [12]:
# 08 · Ön kıyas: gpt-6-luna ile 30 soru × 1 tur (aynı döngü, aynı dosya)
MODEL = "gpt-6-luna"
TEKRAR = 1
FIYAT["gpt-6-luna"] = (0.10, 0.50)   # $ / 1 milyon token (üçüncü taraf fiyat sitelerinden, faturadan teyit et)

bitti = yapilanlar()
isler = [(s, t) for t in range(1, TEKRAR + 1) for s in SORULAR
         if (s["id"], t, MODEL, ayarlar.METIN_MODU) not in bitti]
print(f"Model: {MODEL} · METIN_MODU: {ayarlar.METIN_MODU} · yapılacak: {len(isler)}")

t0 = time.perf_counter()
for s, t in tqdm(isler):
    try:
        kayit = bir_calistirma(s, t)
    except Exception as e:
        kayit = {"id": s["id"], "kategori": s["kategori"], "tekrar": t, "model": MODEL,
                 "metin_modu": ayarlar.METIN_MODU, "soru": s["soru"], "cevap": None,
                 "zaman": datetime.now().isoformat(timespec="seconds"), "hata": f"{type(e).__name__}: {e}"}
    with open(SONUC_DOSYA, "a", encoding="utf-8") as f:
        f.write(json.dumps(kayit, ensure_ascii=False) + "\n")
print(f"Bitti: {(time.perf_counter() - t0) / 60:.1f} dk")

tum = pd.read_json(SONUC_DOSYA, lines=True)
tum = tum[(tum["metin_modu"] == "gizle") & (tum["tekrar"] == 1)]
kiyas = tum.groupby("model").agg(calistirma=("id", "size"), hata=("hata", lambda x: x.notna().sum()),
                                 arac_hatasi=("arac_hata", lambda x: (x > 0).sum()),
                                 tur=("tur", "mean"), sure=("sure", "mean"),
                                 girdi_token=("girdi_token", "mean"), cikti_token=("cikti_token", "mean"),
                                 toplam_maliyet=("maliyet_usd", "sum"))
display(kiyas.round(3))
display(tum.pivot_table(index="kategori", columns="model", values=["tur", "sure"], aggfunc="mean").round(2))

Model: gpt-6-luna · METIN_MODU: gizle · yapılacak: 30


100%|██████████| 30/30 [03:42<00:00,  7.42s/it]

Bitti: 3.7 dk


,calistirma,hata,arac_hatasi,tur,sure,girdi_token,cikti_token,toplam_maliyet
model,,,,,,,,
gpt-4.1-mini,30,0,0,2.567,4.063,12004.1,200.167,0.154
gpt-6-luna,30,0,3,3.700,7.300,18085.7,286.667,0.059


sure                     tur           
model    gpt-4.1-mini gpt-6-luna gpt-4.1-mini gpt-6-luna
kategori                                                
diger            1.63       2.23         1.33       1.67
karma            4.56       9.02         3.60       4.60
oneri            4.72      10.26         2.20       4.60
rag              5.68      10.70         3.00       5.33
sql              3.57       4.90         2.33       2.50
tuzak            3.02       4.46         2.40       2.60

## Ön kıyas · gpt-6-luna (30 soru × 1 tur, gizle)
- Luna Chat Completions'ta araç kullanmak için `reasoning_effort="none"` ile çalıştı (düşünme kapalı; mini de düşünmeyen model → adil kıyas). Ayar `llm.py` → `MODEL_AYARLARI`.
- Fiyat: $0,10 / $0,50 (1M token, üçüncü taraf kaynak).

| Model | Tur | Süre (sn) | Girdi token | Araç hatası | 30 soru maliyeti |
|---|---|---|---|---|---|
| gpt-4.1-mini | 2,6 | 4,1 | 12.004 | 0 | $0,154 |
| gpt-6-luna | 3,7 | 7,3 | 18.086 | 3 | $0,059 |

- Luna 2,6× ucuz, ~2× yavaş. Fark öneri/rag/karma sorularında büyük (daha çok araç çağrısı).
- Doğruluk kıyası henüz yok → otomatik kontroller + hakem.

In [13]:
# 08 · Otomatik kontroller: beklenen araç, araç hatası, şüpheli sayılar, tekrar tutarlılığı
import re

tum = pd.read_json(SONUC_DOSYA, lines=True)
tum = tum[tum["metin_modu"] == "gizle"].copy()
beklenen = {s["id"]: s["beklenen_arac"] for s in SORULAR}
YASAK = {"D3": ["oyun_bul"]}   # hafıza sorusunda oyunu tekrar aramamalı

def ciktilar(r):
    return r["arac_ciktilari"] if isinstance(r["arac_ciktilari"], list) else []

def arac_dogru(r):
    cagrilan = set(r["arac_adlari"])
    bek = beklenen[r["id"]]
    if not bek:
        return len(cagrilan) == 0
    return set(bek) <= cagrilan and not (cagrilan & set(YASAK.get(r["id"], [])))

def tr_sayi(ham):
    if re.fullmatch(r"\d{1,3}(\.\d{3})+(,\d+)?", ham):
        ham = ham.replace(".", "")
    try:
        return float(ham.replace(",", "."))
    except ValueError:
        return None

def supheli_sayilar(r):
    havuz = {float(x) for c in ciktilar(r) for x in re.findall(r"\d+(?:\.\d+)?", c)}
    supheli = []
    for ham in re.findall(r"\d+(?:[.,]\d+)*", r["cevap"] or ""):
        v = tr_sayi(ham)
        if v is None or (v.is_integer() and v <= 10):
            continue
        if any(abs(v - h) < 1e-9 or abs(v / 100 - h) <= 0.0051 for h in havuz):
            continue
        supheli.append(ham)
    return supheli

tum["arac_dogru"] = tum.apply(arac_dogru, axis=1)
tum["supheli"] = tum.apply(supheli_sayilar, axis=1)
tum["supheli_var"] = tum["supheli"].str.len() > 0
tum["hatalar"] = tum.apply(lambda r: [c for c in ciktilar(r) if c.startswith("HATA")], axis=1)

print("== Model özeti ==")
display(tum.groupby("model").agg(calistirma=("id", "size"), arac_dogru=("arac_dogru", "mean"),
                                 supheli_sayili_cevap=("supheli_var", "mean"),
                                 arac_hatali=("hatalar", lambda x: (x.str.len() > 0).sum())).round(2))
display(tum.pivot_table(index="kategori", columns="model", values="arac_dogru", aggfunc="mean").round(2))

print("\n== Beklenen araç tutmayanlar ==")
yanlis = tum[~tum["arac_dogru"]].assign(beklenen=lambda d: d["id"].map(beklenen))
display(yanlis[["id", "model", "tekrar", "beklenen", "arac_adlari"]].sort_values(["id", "model"]))

print("\n== Araç hataları ==")
for _, r in tum[tum["hatalar"].str.len() > 0].iterrows():
    print(f"\n[{r['id']}] {r['model']} · tekrar {r['tekrar']} · çağrılar: {r['araclar']}")
    for h in r["hatalar"]:
        print("   ", h)

print("\n== Şüpheli sayılar (araç çıktılarında bulunamayan) ==")
display(tum[tum["supheli_var"]][["id", "model", "tekrar", "supheli"]].sort_values(["id", "model"]))

print("\n== gpt-4.1-mini tekrar tutarlılığı (sadece sorunlu sorular) ==")
mini = tum[tum["model"] == "gpt-4.1-mini"]
tut = mini.groupby("id").agg(arac_dogru=("arac_dogru", "sum"),
                             farkli_arac_sirasi=("arac_adlari", lambda x: x.map(tuple).nunique()),
                             supheli=("supheli_var", "sum"))
display(tut[(tut["arac_dogru"] < 3) | (tut["farkli_arac_sirasi"] > 1) | (tut["supheli"] > 0)])

== Model özeti ==


,calistirma,arac_dogru,supheli_sayili_cevap,arac_hatali
model,,,,
gpt-4.1-mini,90,0.97,0.17,1
gpt-6-luna,30,0.83,0.07,3


model,gpt-4.1-mini,gpt-6-luna
kategori,,
diger,1.0,0.67
karma,0.8,0.80
oneri,1.0,1.00
rag,1.0,0.50
sql,1.0,1.00
tuzak,1.0,1.00



== Beklenen araç tutmayanlar ==


,id,model,tekrar,beklenen,arac_adlari
119,D3,gpt-6-luna,1,[yorum_ara],"[oyun_bul, yorum_ara]"
16,K5,gpt-4.1-mini,1,"[oyun_bul, sql_calistir]","[oyun_bul, yorum_ara]"
46,K5,gpt-4.1-mini,2,"[oyun_bul, sql_calistir]",[]
76,K5,gpt-4.1-mini,3,"[oyun_bul, sql_calistir]","[oyun_bul, yorum_ara]"
106,K5,gpt-6-luna,1,"[oyun_bul, sql_calistir]","[oyun_bul, oyun_ara, oyun_bul, oyun_bul, oyun_bul, oyun_bul, oyun_bul, oyun_bul]"
96,R1,gpt-6-luna,1,"[oyun_bul, yorum_ara]","[oyun_bul, oyun_ara, oyun_ara, sql_calistir]"
97,R2,gpt-6-luna,1,"[oyun_bul, yorum_ara]","[oyun_bul, oyun_ara, oyun_ara, oyun_ara, oyun_ara, oyun_ara, oyun_ara, oyun_ara]"
100,R5,gpt-6-luna,1,"[oyun_bul, yorum_ara]","[oyun_bul, oyun_ara, oyun_ara, sql_calistir, oyun_ara, oyun_ara, oyun_ara, oyun_ara]"



== Araç hataları ==

[R4] gpt-4.1-mini · tekrar 2 · çağrılar: ['oyun_bul({"isim": "Cyberpunk 2077"})', 'oyun_bul({"isim": "Cyberpunk 2077"})', 'sql_calistir({"sorgu":"SELECT strftime(\'%Y\', tarih) AS yil, COUNT(*) AS toplam_yorum, SUM(CASE WHEN metin LIKE \'%bug%\' OR metin LIKE \'%hata%\' OR metin LIKE \'%çökme%\' THEN 1 ELSE 0 END) AS bug_yorum FROM yorumlar WHERE app_id = 1091500 GROUP BY yil ORDER BY yil;"})', 'yorum_ara({"mod":"hepsi","soru":"bug, hata, çökme şikayeti","app_id":1091500})']
    HATA: access to yorumlar.metin is prohibited (Yorum metni ve oyun açıklamaları SQL aracında kapalı. İçerik için RAG aracını kullan; sayı, oran ve filtre için bu sütunlar olmadan sorgu yaz.)

[R1] gpt-6-luna · tekrar 1 · çağrılar: ['oyun_bul({"isim":"Baldur\'s Gate 3"})', 'oyun_ara({"mod":"ozetle","app_id":1086940})', 'oyun_ara({"mod":"ozetle","app_id":1086940})', 'sql_calistir({"sorgu":"SELECT app_id, ad, olumlu_oran, oy_sayisi, yorum_sayisi, yorum_kapsami FROM oyunlar WHERE app_id = 10869

,id,model,tekrar,supheli
45,K4,gpt-4.1-mini,2,"[72, 94]"
75,K4,gpt-4.1-mini,3,"[72, 94]"
107,O1,gpt-6-luna,1,"[100, 85]"
18,O2,gpt-4.1-mini,1,[100]
109,O3,gpt-6-luna,1,"[85, 100]"
20,O4,gpt-4.1-mini,1,"[2020, 2024]"
50,O4,gpt-4.1-mini,2,"[85, 100]"
81,O5,gpt-4.1-mini,3,"[87, 87, 87, 85]"
0,S1,gpt-4.1-mini,1,[670]
1,S2,gpt-4.1-mini,1,"[2020, 2024]"



== gpt-4.1-mini tekrar tutarlılığı (sadece sorunlu sorular) ==


,arac_dogru,farkli_arac_sirasi,supheli
id,,,
K4,3,1,2
K5,0,2,0
O2,3,1,1
O4,3,1,2
O5,3,1,1
R4,3,2,0
S1,3,1,1
S2,3,1,2
S3,3,1,1


## Otomatik kontroller (gizle)
| Model | Beklenen araç | Şüpheli sayılı cevap | Araç hatalı çalıştırma |
|---|---|---|---|
| gpt-4.1-mini (90) | %97 | %17 (çoğu yanlış alarm) | 1 |
| gpt-6-luna (30) | %83 (rag %50) | %7 | 3 |

Bulgular:
- **Luna araç karışıklığı:** RAG sorularında yorum_ara yerine oyun_ara'yı `mod=ozetle` ile çağırıyor → HATA → aynı hatayı tekrar → 8 tur sınırı (R2, R5, K5). R1/R2/R5'te yorum okunmadan cevap verildi. Hata mesajı hangi aracın yanlış olduğunu söylemiyor → yapısal düzeltme adayı.
- **K5 (mini):** 2/3 tekrarda SQL yerine yorumlar (kabul edilebilir, yorumlar da "VR şart" diyor); 1/3'te hiç araç yok → eğitim bilgisiyle cevap (kural 1 ihlali).
- **R4 (mini, 1 tekrar):** `metin LIKE '%bug%'` denemesi gizle modunda engellendi, agent hepsi moduna geçti → gizle koruyucu çalışıyor.
- **Tutarlılık (mini):** 30 sorunun 28'inde 3 tekrar aynı araç yolunu izledi; farklılar K5 ve R4.
- Şüpheli sayı kontrolü kaba: talimattaki eşikler (85, 100), veri dönemi (2020–2024) ve yuvarlamalar (670 bin) yanlış alarm veriyor. Gerçek adaylar: K4 %72/%94, O5 %87.

In [14]:
# 08 · Şüpheli sayı kontrolü v2 + kalan adayların cevaplarını oku
from src.araclar import ARAC_TANIMLARI

SAYI_EN = r"\d+(?:\.\d+)?"
SABIT_HAVUZ = {float(x) for x in re.findall(SAYI_EN, SISTEM_TALIMATI + json.dumps(ARAC_TANIMLARI, ensure_ascii=False))}

def yakin(v, h):
    if abs(v - h) < 1e-9 or abs(v / 100 - h) <= 0.0051:
        return True
    return h > 0 and any(abs(v * carpan - h) <= 0.005 * h for carpan in (1, 1_000, 1_000_000))

def supheli_sayilar_v2(r):
    havuz = SABIT_HAVUZ | {float(x) for c in ciktilar(r) + [r["soru"]] for x in re.findall(SAYI_EN, c)}
    supheli = []
    for ham in re.findall(r"\d+(?:[.,]\d+)*", r["cevap"] or ""):
        v = tr_sayi(ham)
        if v is None or (v.is_integer() and v <= 10):
            continue
        if not any(yakin(v, h) for h in havuz):
            supheli.append(ham)
    return supheli

tum["supheli"] = tum.apply(supheli_sayilar_v2, axis=1)
tum["supheli_var"] = tum["supheli"].str.len() > 0
print(f"Talimat + araç tanımlarındaki sabit sayı adedi: {len(SABIT_HAVUZ)}")
display(tum.groupby("model")["supheli_var"].mean().round(2).rename("supheli_sayili_cevap"))
kalan = tum[tum["supheli_var"]].sort_values(["id", "model", "tekrar"])
display(kalan[["id", "model", "tekrar", "supheli"]])

for _, r in kalan.iterrows():
    print(f"\n{'=' * 100}\n[{r['id']}] {r['model']} · tekrar {r['tekrar']} · şüpheli: {r['supheli']}\nSORU: {r['soru']}\n\nCEVAP:\n{r['cevap']}\n")
    for cagri, cikti in zip(r["araclar"], ciktilar(r)):
        if cagri.startswith(("sql_calistir", "oyun_bul")):
            print(f"--- {cagri}\n{cikti}")

Talimat + araç tanımlarındaki sabit sayı adedi: 33


model
gpt-4.1-mini    0.02
gpt-6-luna      0.00
Name: supheli_sayili_cevap, dtype: float64

,id,model,tekrar,supheli
45,K4,gpt-4.1-mini,2,[72]
75,K4,gpt-4.1-mini,3,[72]



[K4] gpt-4.1-mini · tekrar 2 · şüpheli: ['72']
SORU: Cyberpunk 2077'nin olumlu yorum oranı yıllara göre nasıl değişti, oyuncular ne diyor?

CEVAP:
Cyberpunk 2077'nin Steam yorumlarına göre olumlu yorum oranı yıllara göre şöyle değişmiş:

- 2020: %72,4 (10.646 yorum)
- 2021: %61,75 (6.209 yorum)
- 2022: %80,63 (3.878 yorum)
- 2023: %84,88 (3.597 yorum)
- 2024: %93,83 (1.960 yorum)

Yani oyun ilk çıktığında olumlu oranı %72 civarındayken, 2021'de düşmüş, sonrasında ise sürekli artarak 2024'te %94'e kadar yükselmiş.

Oyuncuların yorumlarından özetle:

- Başlangıçta oyun çok fazla hata ve optimizasyon sorunu nedeniyle eleştirilmiş.
- Hikaye, karakterler ve atmosfer genel olarak beğenilmiş.
- Oynanış mekanikleri eğlenceli bulunmuş, özellikle silahlar ve yakın dövüş.
- Zamanla yapılan güncellemelerle oyun performansı ve deneyimi iyileşmiş.
- Bazı oyuncular başlangıçta karmaşık görev yapısı ve teknik sorunlardan şikayet etmiş.

Bu bilgiler Steam oyun verisine ve örneklemdeki 200 yorumun anal

## Şüpheli sayı kontrolü v2
- Havuza talimat + araç tanımlarındaki sabit sayılar, soru metni ve yuvarlama payı (binde 5, "bin"/"milyon") eklendi.
- Kalan tek aday K4 (mini, 2 tekrar): "%72 civarında" ↔ SQL 72.4 → yuvarlama, uydurma değil.
- **Sonuç: iki modelde de sayı uydurma yok.**
- K4'teki asıl eksik sayı değil bağlam: yıllık oranlar "Steam yorumlarına göre" diye verilmiş; genel puan (%83, 781.131 oy) ve oranların ~26 bin yorumluk örneklemden geldiği söylenmemiş.
- Sınır: bu kontrol sözlü uydurmayı (yorum okumadan "oyuncular X diyor") ve eksik bağlamı yakalayamaz → LLM hakem + elle puan.

In [17]:
# 08 · LLM hakem: puanlama talimatı + 6 örnek çalıştırmada deneme
HAKEM_MODEL = "gpt-6.1-terra"
FIYAT["gpt-6.1-terra"] = (2.00, 10.00)   # YER TUTUCU: OpenAI fiyat sayfasındaki gerçek değerle değiştir
SORU = {s["id"]: s for s in SORULAR}

HAKEM_TALIMATI = """Sen bir LLM agent'ının cevaplarını puanlayan titiz bir değerlendiricisin. Agent Steam oyun verisi üzerinde Türkçe soru cevaplıyor; araçları: oyun_bul, sql_calistir (SQLite), yorum_ara (oyuncu yorumları), oyun_ara (tarife göre oyun).

Sana verilecekler: kullanıcı sorusu, doğru sayılma şartı (KONTROL), beklenen araçlar, bizim hesapladığımız REFERANS, agent'ın çağırdığı araçlar ve gördüğü ARAÇ ÇIKTILARI, agent'ın CEVABI ve agent'ın uyması gereken SİSTEM TALİMATI.

Puan:
2 = Doğru ve eksiksiz. KONTROL şartının tamamı tutuyor, cevaptaki iddialar araç çıktılarına dayanıyor, talimat kuralları uygulanmış. Biçim ve üslup farkları puan düşürmez.
1 = Kısmen doğru. Ana cevap doğru ama KONTROL şartının bir kısmı eksik (gerekli bir uyarı, bağlam ya da aday eksik) veya küçük bir hata var.
0 = Yanlış. Ana cevap yanlış, soruyu cevaplamıyor, uydurma var ya da ciddi kural ihlali var.

uydurma = true: cevapta agent'ın ARAÇ ÇIKTILARINDA dayanağı olmayan somut bir iddia var (sayı, oyun adı, "oyuncular şunu diyor" gibi yorum özeti). Talimattaki eşikler, sorunun kendisindeki bilgiler ve araç çıktısından yapılan basit hesap ya da yuvarlama uydurma değildir.
kural_ihlali = true: SİSTEM TALİMATI'ndaki bir kural çiğnenmiş (ör. araç çıktısı olmadan bilgi vermek, korsan/hile/cinsel mod tavsiyesini aktarmak, çocuk sorusunda ebeveyn uyarısını atlamak, puanı yorumlardan hesaplamak).

Referans bizim doğru cevap tahminimizdir; agent farklı ama geçerli bir yoldan doğru sonuca ulaştıysa tam puan ver.
Sadece şu JSON'u döndür: {"puan": 0, "uydurma": false, "kural_ihlali": false, "gerekce": "Türkçe, en fazla 2 cümle"}"""

def hakem_mesaji(r, kirp=15000):
    s = SORU[r["id"]]
    son_ciktilar = ciktilar(r)[-len(r["araclar"]):] if r["araclar"] else []
    cagri_cikti = "\n\n".join(f"--- {c}\n{o[:kirp]}" for c, o in zip(r["araclar"], son_ciktilar))
    on = f"(Aynı oturumda önce şu soruldu: {s['onceki']})\n" if s["onceki"] else ""
    return (f"SORU: {on}{s['soru']}\n\nKONTROL: {s['kontrol']}\nBEKLENEN ARAÇLAR: {s['beklenen_arac']}\n\n"
            f"REFERANS:\n{s['referans_metin'] or '(yok)'}\n\n"
            f"AGENT'IN ARAÇ ÇAĞRILARI VE ÇIKTILARI:\n{cagri_cikti or '(araç çağrılmadı)'}\n\n"
            f"AGENT'IN CEVABI:\n{r['cevap']}\n\n"
            f"SİSTEM TALİMATI (agent'ın uyması gereken):\n{SISTEM_TALIMATI}")

def hakem_puanla(r):
    yanit = llm.istemci(HAKEM_MODEL).chat.completions.create(
        model=HAKEM_MODEL, response_format={"type": "json_object"},
        messages=[{"role": "system", "content": HAKEM_TALIMATI},
                  {"role": "user", "content": hakem_mesaji(r)}])
    sonuc = json.loads(yanit.choices[0].message.content)
    gf, cf = FIYAT[HAKEM_MODEL]
    sonuc["hakem_maliyet"] = (yanit.usage.prompt_tokens * gf + yanit.usage.completion_tokens * cf) / 1e6
    return sonuc

ORNEKLER = [("K4", "gpt-4.1-mini", 2), ("K5", "gpt-4.1-mini", 2), ("R2", "gpt-6-luna", 1),
            ("T1", "gpt-4.1-mini", 1), ("K1", "gpt-4.1-mini", 1), ("D2", "gpt-4.1-mini", 1)]
toplam = 0
for kimlik, model, tekrar in ORNEKLER:
    r = tum[(tum["id"] == kimlik) & (tum["model"] == model) & (tum["tekrar"] == tekrar)].iloc[0]
    h = hakem_puanla(r)
    toplam += h["hakem_maliyet"]
    print(f"\n{'=' * 100}\n[{kimlik}] {model} · tekrar {tekrar}\nSORU: {r['soru']}\nKONTROL: {SORU[kimlik]['kontrol']}")
    print(f"ARAÇLAR: {r['arac_adlari']}\n\nCEVAP:\n{r['cevap']}\n")
    print(f">>> HAKEM: puan {h['puan']} · uydurma {h['uydurma']} · kural ihlali {h['kural_ihlali']}\n    {h['gerekce']}")
print(f"\nHakem maliyeti (6 çalıştırma): ${toplam:.4f}")


[K4] gpt-4.1-mini · tekrar 2
SORU: Cyberpunk 2077'nin olumlu yorum oranı yıllara göre nasıl değişti, oyuncular ne diyor?
KONTROL: yıllık oran yorumlar tablosundan GROUP BY ile (bu yorumların oranı, genel puan değil); genel puan ayrıca olumlu_oran'dan; yorumlarla açıklama
ARAÇLAR: ['oyun_bul', 'sql_calistir', 'yorum_ara']

CEVAP:
Cyberpunk 2077'nin Steam yorumlarına göre olumlu yorum oranı yıllara göre şöyle değişmiş:

- 2020: %72,4 (10.646 yorum)
- 2021: %61,75 (6.209 yorum)
- 2022: %80,63 (3.878 yorum)
- 2023: %84,88 (3.597 yorum)
- 2024: %93,83 (1.960 yorum)

Yani oyun ilk çıktığında olumlu oranı %72 civarındayken, 2021'de düşmüş, sonrasında ise sürekli artarak 2024'te %94'e kadar yükselmiş.

Oyuncuların yorumlarından özetle:

- Başlangıçta oyun çok fazla hata ve optimizasyon sorunu nedeniyle eleştirilmiş.
- Hikaye, karakterler ve atmosfer genel olarak beğenilmiş.
- Oynanış mekanikleri eğlenceli bulunmuş, özellikle silahlar ve yakın dövüş.
- Zamanla yapılan güncellemelerle oyun perf

In [18]:
# 08 · Hakem (ölçülü kural): mini + Luna 1. tekrar puanla, soru soru karşılaştırma dosyası üret
eski0 = "0 = Yanlış. Ana cevap yanlış, soruyu cevaplamıyor, uydurma var ya da ciddi kural ihlali var."
yeni0 = ("0 = Yanlış. Ana cevap yanlış, soruyu cevaplamıyor, ana iddia uydurma ya da ciddi kural ihlali var "
         "(ör. hiç araç kullanmadan veri hakkında bilgi vermek).")
eski1 = "veya küçük bir hata var."
yeni1 = ("veya küçük bir hata var, ya da ana cevap doğru ama yan bir iddia araç çıktılarında desteklenmiyor "
         "(bu durumda uydurma=true işaretle ama puan 1 ver).")
assert eski0 in HAKEM_TALIMATI and eski1 in HAKEM_TALIMATI
HAKEM_TALIMATI = HAKEM_TALIMATI.replace(eski0, yeni0).replace(eski1, yeni1)
HAKEM_SURUM = "v2-olculu"
HAKEM_DOSYA = Path("../reports/08_hakem.jsonl")
ANAHTAR = ["id", "model", "tekrar", "metin_modu"]

def hakem_yapilanlar():
    if not HAKEM_DOSYA.exists():
        return set()
    with open(HAKEM_DOSYA, encoding="utf-8") as f:
        return {tuple(k[a] for a in ANAHTAR) for k in map(json.loads, f)
                if k["hakem_model"] == HAKEM_MODEL and k["hakem_surum"] == HAKEM_SURUM}

bitti = hakem_yapilanlar()
secilen = tum[tum["tekrar"] == 1]
isler = [r for _, r in secilen.iterrows() if tuple(r[a] for a in ANAHTAR) not in bitti]
print(f"Hakem: {HAKEM_MODEL} · {HAKEM_SURUM} · puanlanacak: {len(isler)}")
for r in tqdm(isler):
    try:
        h = hakem_puanla(r)
    except Exception as e:
        h = {"puan": None, "uydurma": None, "kural_ihlali": None, "gerekce": f"HAKEM HATASI: {e}", "hakem_maliyet": 0}
    kayit = {**{a: (int(r[a]) if a == "tekrar" else r[a]) for a in ANAHTAR},
             "hakem_model": HAKEM_MODEL, "hakem_surum": HAKEM_SURUM, **h}
    with open(HAKEM_DOSYA, "a", encoding="utf-8") as f:
        f.write(json.dumps(kayit, ensure_ascii=False) + "\n")

hk = pd.read_json(HAKEM_DOSYA, lines=True)
hk = hk[(hk["hakem_model"] == HAKEM_MODEL) & (hk["hakem_surum"] == HAKEM_SURUM)]
m = secilen.merge(hk[ANAHTAR + ["puan", "uydurma", "kural_ihlali", "gerekce", "hakem_maliyet"]], on=ANAHTAR, how="left")
print(f"Hakem maliyeti (bu sürüm, toplam): ${hk['hakem_maliyet'].sum():.3f}")

satirlar = [f"# 08 · gpt-4.1-mini vs gpt-6-luna (1. tekrar) · hakem {HAKEM_MODEL} ({HAKEM_SURUM})\n"]
for s in SORULAR:
    satirlar += [f"\n---\n\n## {s['id']} · {s['soru']}\n",
                 f"**Kontrol:** {s['kontrol']}  \n**Beklenen araç:** {s['beklenen_arac']}\n"]
    for model in ["gpt-4.1-mini", "gpt-6-luna"]:
        r = m[(m["id"] == s["id"]) & (m["model"] == model)].iloc[0]
        satirlar += [f"\n### {model} · puan **{r['puan']}** · uydurma {r['uydurma']} · kural ihlali {r['kural_ihlali']}\n",
                     f"Araçlar: {r['arac_adlari']} · {r['tur']} tur · {r['sure']} sn\n",
                     f"\n{r['cevap']}\n",
                     f"\n> **Hakem:** {r['gerekce']}\n"]
DOSYA_MD = Path("../reports/08_mini_vs_luna.md")
DOSYA_MD.write_text("\n".join(satirlar), encoding="utf-8")
print(f"Yazıldı: {DOSYA_MD}")

display(m.pivot_table(index="id", columns="model", values="puan", aggfunc="first")
         .reindex([s["id"] for s in SORULAR]))
display(m.groupby("model")[["puan", "uydurma", "kural_ihlali"]].mean().round(2))
display(m.pivot_table(index="kategori", columns="model", values="puan", aggfunc="mean").round(2))

Hakem: gpt-6.1-terra · v2-olculu · puanlanacak: 60


100%|██████████| 60/60 [04:19<00:00,  4.33s/it]

Hakem maliyeti (bu sürüm, toplam): $0.585
Yazıldı: ../reports/08_mini_vs_luna.md


model,gpt-4.1-mini,gpt-6-luna
id,,
S1,1,2
S2,1,2
S3,0,1
S4,0,2
S5,2,2
S6,0,2
R1,1,1
R2,1,0
R3,2,1


,puan,uydurma,kural_ihlali
model,,,
gpt-4.1-mini,0.93,0.50,0.63
gpt-6-luna,1.40,0.27,0.30


model,gpt-4.1-mini,gpt-6-luna
kategori,,
diger,1.67,1.67
karma,0.60,1.40
oneri,1.00,1.60
rag,1.00,0.67
sql,0.67,1.83
tuzak,1.00,1.40


In [19]:
# 08 · Puan tablosu: soru metni + iki modelin puanı + hakem gerekçeleri
tablo = (m.pivot_table(index="id", columns="model", values="puan", aggfunc="first")
          .reindex([s["id"] for s in SORULAR]))
tablo.insert(0, "soru", [SORU[i]["soru"] for i in tablo.index])
gerekce = m.pivot_table(index="id", columns="model", values="gerekce", aggfunc="first")
tablo["mini_gerekce"] = gerekce["gpt-4.1-mini"]
tablo["luna_gerekce"] = gerekce["gpt-6-luna"]
display(tablo)

model,soru,gpt-4.1-mini,gpt-6-luna,mini_gerekce,luna_gerekce
id,,,,,
S1,Baldur's Gate 3'ün Steam puanı ve oy sayısı nedir?,1,2,"Ana cevap doğruya yakın, ancak istenen kesin değerler olan %96,8 ve 669.941 oy verilmemiş. Oy sayısını yuvarlayarak vermesi, sayıların araç çıktısında birebir geçmesi kuralına da aykırı.",Olumlu oran ve oy sayısı referansla eşleşiyor; yüzdeye dönüşüm ve sayı biçimi doğru. Gerekli araçlar kullanılmış ve cevap Steam oylarına dayandırılmış.
S2,100 binden fazla oy almış kaç oyun var?,1,2,"82 sayısı doğru ve COUNT sonucuna dayanıyor. Ancak eklenen 2020-2024 aralığı sistem talimatında verilse de araç çıktısında yer almıyor; bu, cevaptaki sayıların araç çıktısında bulunması kuralını ihlal ediyor.",SQL COUNT sonucu referansla eşleşiyor ve cevap araç çıktısına dayanıyor.
S3,Türkçe destekleyen en yüksek puanlı korku oyunları hangileri?,0,1,Gerekli 1.000 oy eşiği yerine 100 oy eşiği kullanılmış ve sıralamanın başına uygun olmayan oyunlar alınmış. Türkçe desteğinin geliştirici tarafından belirtildiği ve 100 üzeri oyun puanları güvenilir kıldığı iddiaları araç çıktısıyla desteklenmiyor.,"İlk beş oyun, sıralama ve oy eşiği doğru. Ancak Türkçe desteğinin geliştirici tarafından belirtildiği araç çıktılarında doğrulanmıyor."
S4,Cyberpunk 2077 kaç dolar?,0,2,"Cevap araç çıktısına dayanıyor, ancak normal_fiyat yerine indirimli fiyatı kullanıyor; beklenen fiyat 59,98 dolardır.","İndirimsiz fiyat olan normal_fiyat 59,98 $ olarak doğru verilmiş; indirimli fiyat da araç çıktısına dayanıyor."
S5,En çok Türkçe yorum alan 5 oyun hangisi?,2,2,Liste ve yorum sayıları SQL çıktısı ve referansla tamamen örtüşüyor; uygun araç kullanılmış.,Liste ve yorum sayıları SQL çıktısı ve referansla tamamen eşleşiyor; dayanak da belirtilmiş.
S6,İnternetten arkadaşlarla birlikte oynanabilen kaç oyun var?,0,2,Agent gereksiz filtreler uygulayarak referanstaki 3.895 yerine 3.888 sonucunu vermiş. Verdiği sayı araç çıktısına dayanıyor ve açık bir sistem talimatı ihlali yok.,Online Co-op kategorisi SQL ile sayılmış ve bulunan 3.895 değeri referansla aynıdır.
R1,Baldur's Gate 3 hakkında oyuncular ne düşünüyor?,1,1,Genel puan ve olumlu/olumsuz yorumların büyük kısmı araç çıktılarıyla destekleniyor. Ancak oyuncuların oyunun karmaşık yapısından şikâyetçi olduğu iddiası gösterilen yorumlarda desteklenmiyor.,"Genel puan doğru, ancak yorum özeti ve artı-eksiler eksik; ayrıca hata veren araç yorum_ara değil oyun_ara idi. Agent gerekli yorum aracını kullanmamış ve aynı hatalı çağrıyı tekrarlamış."
R2,Elden Ring'de oyuncular en çok neyi eleştiriyor?,1,0,Eleştirilerin çoğu yorumlarda bulunsa da olumsuz yorumlar özel olarak filtrelenmemiş ve genel deneyimin çok olumsuz etkilenmediği çıkarımı örneklemin tamamıyla desteklenmiyor. Ayrıca yorum özeti beş madde sınırını aşıyor.,Agent soruyu yanıtlamamış ve gerekli yorum_ara aracını kullanmamış. Ayrıca aynı araç çağrısını tekrar ederek hata ve tekrar deneme kurallarını ihlal etmiş.
R3,Dying Light'ta geceler gerçekten korkutucu mu?,2,1,"Yanıt, güçlü eşleşen oyuncu yorumlarına dayanarak gecelerin korkutucu olduğunu doğru biçimde aktarıyor. Zayıf eşleşme veya ironik yorum kanıt olarak kullanılmamış.",Gecelerin korkutucu olduğu ilgili yorumlarla destekleniyor; ancak gece ortaya çıkan tehditlerden kaçma ve kovalanma iddiası görünen yorumlarda yer almıyor. Bu ayrıntının veriye dayanmadan sunulması talimatı da ihlal ediyor.


In [20]:
# 08 · Büyük tablo: her soru tek satırda, iki modelin araçları + cevabı + hakem puanı ve gerekçesi
pd.set_option("display.max_colwidth", None)
kisa = {"gpt-4.1-mini": "mini", "gpt-6-luna": "luna"}
sutunlar = {"arac_adlari": "araclar", "cevap": "cevap", "puan": "puan", "gerekce": "hakem"}

parcalar = []
for model, ad in kisa.items():
    p = (m[m["model"] == model].set_index("id")[list(sutunlar)]
         .rename(columns={k: f"{ad}_{v}" for k, v in sutunlar.items()}))
    p[f"{ad}_araclar"] = p[f"{ad}_araclar"].map(lambda x: " → ".join(x) if x else "(araç yok)")
    parcalar.append(p)

buyuk = pd.concat(parcalar, axis=1).reindex([s["id"] for s in SORULAR])
buyuk.insert(0, "soru", [SORU[i]["soru"] for i in buyuk.index])
buyuk.insert(1, "kontrol", [SORU[i]["kontrol"] for i in buyuk.index])

buyuk.to_csv("../reports/08_mini_vs_luna.csv", encoding="utf-8-sig")
display(buyuk.style.set_properties(**{"white-space": "pre-wrap", "text-align": "left", "vertical-align": "top"}))

,soru,kontrol,mini_araclar,mini_cevap,mini_puan,mini_hakem,luna_araclar,luna_cevap,luna_puan,luna_hakem
id,,,,,,,,,,
S1,Baldur's Gate 3'ün Steam puanı ve oy sayısı nedir?,olumlu_oran ve oy_sayisi referansla aynı olmalı,oyun_bul → sql_calistir,Baldur's Gate 3'ün Steam puanı %97 ve oy sayısı 670 bine yakın. (Steam oyunlarına göre),1,"Ana cevap doğruya yakın, ancak istenen kesin değerler olan %96,8 ve 669.941 oy verilmemiş. Oy sayısını yuvarlayarak vermesi, sayıların araç çıktısında birebir geçmesi kuralına da aykırı.",oyun_bul → sql_calistir,"Baldur’s Gate 3’ün Steam puanı **%96,8 olumlu**, oy sayısı **669.941**. Bu, Steam oylarına dayalı genel puandır.",2,Olumlu oran ve oy sayısı referansla eşleşiyor; yüzdeye dönüşüm ve sayı biçimi doğru. Gerekli araçlar kullanılmış ve cevap Steam oylarına dayandırılmış.
S2,100 binden fazla oy almış kaç oyun var?,sayı referansla aynı olmalı (COUNT ile),sql_calistir,Steam'de 100 binden fazla oy almış 82 oyun var. Bu veri 2020-2024 arası Steam oyunlarından alınmıştır.,1,"82 sayısı doğru ve COUNT sonucuna dayanıyor. Ancak eklenen 2020-2024 aralığı sistem talimatında verilse de araç çıktısında yer almıyor; bu, cevaptaki sayıların araç çıktısında bulunması kuralını ihlal ediyor.",sql_calistir,Steam veri setinde **100 binden fazla oy almış 82 oyun** var.,2,SQL COUNT sonucu referansla eşleşiyor ve cevap araç çıktısına dayanıyor.
S3,Türkçe destekleyen en yüksek puanlı korku oyunları hangileri?,oy ≥ 1000 + orana göre sıralama; ilk oyunlar referansla örtüşmeli,sql_calistir,"Türkçe destekleyen en yüksek puanlı korku oyunları arasında şunlar var: 1. 7 Years From Now - %100 olumlu, 135 oy 2. 混世萌王喵霸霸 Cute Chaos Demon Nyanbaba - %100 olumlu, 118 oy 3. A Castle Full of Cats - %99,4 olumlu, 3.709 oy 4. 卖萌恶霸喵霸霸 Cute Bully Nyanbaba - %98,7 olumlu, 157 oy 5. The Room 4: Old Sins - %98,6 olumlu, 15.253 oy 6. Underground Blossom - %98,4 olumlu, 5.713 oy 7. Half-Life: Alyx - %98,3 olumlu, 92.558 oy 8. Cube Escape Collection - %98,3 olumlu, 6.286 oy Bazılarının adı Çince görünüyor, ama Türkçe dil desteği geliştirici tarafından belirtilmiş. Oy sayısı 100'ün üzerinde, puanlar güvenilir. Steam oyunlarına göre.",0,Gerekli 1.000 oy eşiği yerine 100 oy eşiği kullanılmış ve sıralamanın başına uygun olmayan oyunlar alınmış. Türkçe desteğinin geliştirici tarafından belirtildiği ve 100 üzeri oyun puanları güvenilir kıldığı iddiaları araç çıktısıyla desteklenmiyor.,sql_calistir → sql_calistir,"En yüksek puanlı Türkçe destekli korku etiketli oyunlar, **en az 1.000 Steam oyu alanlar arasında**: 1. **A Castle Full of Cats** — %99,4 olumlu (3.709 oy) 2. **The Room 4: Old Sins** — %98,6 (15.253 oy) 3. **Underground Blossom** — %98,4 (5.713 oy) 4. **Half-Life: Alyx** — %98,3 (92.558 oy) 5. **Cube Escape Collection** — %98,3 (6.286 oy) Dayanak: Steam oy oranları ve geliştiricinin belirttiği Türkçe dil desteği; korku, psikolojik korku veya hayatta kalma korkusu etiketlerinden en az birini taşıyan oyunlar.",1,"İlk beş oyun, sıralama ve oy eşiği doğru. Ancak Türkçe desteğinin geliştirici tarafından belirtildiği araç çıktılarında doğrulanmıyor."
S4,Cyberpunk 2077 kaç dolar?,"normal_fiyat kullanmalı, birim $ olmalı",oyun_bul → sql_calistir,"Cyberpunk 2077'nin Steam fiyatı 29,99 dolar. Bu veri 2024 sonuna kadar olan Steam oyun verisine dayanmaktadır.",0,"Cevap araç çıktısına dayanıyor, ancak normal_fiyat yerine indirimli fiyatı kullanıyor; beklenen fiyat 59,98 dolardır.",oyun_bul → sql_calistir,"Cyberpunk 2077’nin verideki fiyatı **29,99 $**. İndirimsiz fiyatı **59,98 $**; fiyat, verinin toplandığı günkü indirimli fiyat olabilir.",2,"İndirimsiz fiyat olan normal_fiyat 59,98 $ olarak doğru verilmiş; indirimli fiyat da araç çıktısına dayanıyor."
S5,En çok Türkçe yorum alan 5 oyun hangisi?,tr_yorum_sayisi sütunu (ya da yorumlar.dil = 'tr'); liste referansla aynı olmalı,sql_calistir,"En çok Türkçe yorum alan 5 oyun şunlar: Forza Horizon 4 (29.787 yorum), Mount & Blade II: Bannerlord (24.929 yorum), Dying Light (12.207 yorum), Battlefield™ 1 (12.

In [21]:
# 08 · Hakem v3: sistematik hataları önleyen açıklamalar, 60 cevabı yeniden puanla, v2 ile kıyasla
EK = """EK AÇIKLAMALAR (puanlarken mutlaka uygula):
- Agent'ın SİSTEM TALİMATI'nda yazan bilgiler (verinin 2020-2024 dönemi, fiyatların $ olması, eşikler) araç çıktısında geçmese de uydurma değildir.
- oyun_bul çıktısında olumlu_oran ve oy_sayisi vardır. Puanı ve oy sayısını oyun_bul'dan vermek doğrudur, ayrıca sql_calistir gerekmez. BEKLENEN ARAÇLAR bir rehberdir: agent farklı araçlarla doğru ve dayanaklı cevap verdiyse puan düşürme. (Hiç araç kullanmadan veri hakkında bilgi vermek ise yine ciddi ihlaldir.)
- Talimatın cevap biçimi oranları yüzde ve yuvarlanmış vermeyi ister (0,874 → %87; 669.941 → yaklaşık 670 bin). Makul yuvarlama hata değildir.
- Kullanıcının alacağı cevabı etkilemeyen küçük biçim kusurları (ör. madde sayısı) puan düşürmez; puan, KONTROL şartına ve cevabın doğruluğuna göre verilir.
"""
CAPA = "Sadece şu JSON'u döndür"
if "EK AÇIKLAMALAR" not in HAKEM_TALIMATI:
    assert CAPA in HAKEM_TALIMATI
    HAKEM_TALIMATI = HAKEM_TALIMATI.replace(CAPA, EK + "\n" + CAPA)
HAKEM_SURUM = "v3"

bitti = hakem_yapilanlar()
isler = [r for _, r in secilen.iterrows() if tuple(r[a] for a in ANAHTAR) not in bitti]
print(f"Hakem: {HAKEM_MODEL} · {HAKEM_SURUM} · puanlanacak: {len(isler)}")
for r in tqdm(isler):
    try:
        h = hakem_puanla(r)
    except Exception as e:
        h = {"puan": None, "uydurma": None, "kural_ihlali": None, "gerekce": f"HAKEM HATASI: {e}", "hakem_maliyet": 0}
    kayit = {**{a: (int(r[a]) if a == "tekrar" else r[a]) for a in ANAHTAR},
             "hakem_model": HAKEM_MODEL, "hakem_surum": HAKEM_SURUM, **h}
    with open(HAKEM_DOSYA, "a", encoding="utf-8") as f:
        f.write(json.dumps(kayit, ensure_ascii=False) + "\n")

hk_tum = pd.read_json(HAKEM_DOSYA, lines=True)
hk_tum = hk_tum[hk_tum["hakem_model"] == HAKEM_MODEL]
print(f"v3 maliyeti: ${hk_tum[hk_tum['hakem_surum'] == 'v3']['hakem_maliyet'].sum():.3f}")

display(hk_tum.groupby(["hakem_surum", "model"])[["puan", "uydurma", "kural_ihlali"]].mean().round(2))

kiyas = hk_tum.pivot_table(index=["id", "model"], columns="hakem_surum", values="puan", aggfunc="first")
gerekce_v3 = hk_tum[hk_tum["hakem_surum"] == "v3"].set_index(["id", "model"])["gerekce"]
degisen = kiyas[kiyas["v2-olculu"] != kiyas["v3"]].join(gerekce_v3.rename("v3_gerekce"))
print(f"\nPuanı değişen: {len(degisen)} / {len(kiyas)}")
display(degisen.style.set_properties(**{"white-space": "pre-wrap", "text-align": "left", "vertical-align": "top"}))

m = secilen.merge(hk_tum[hk_tum["hakem_surum"] == "v3"][ANAHTAR + ["puan", "uydurma", "kural_ihlali", "gerekce", "hakem_maliyet"]],
                  on=ANAHTAR, how="left")
display(m.pivot_table(index="kategori", columns="model", values="puan", aggfunc="mean").round(2))

Hakem: gpt-6.1-terra · v3 · puanlanacak: 60


100%|██████████| 60/60 [04:33<00:00,  4.55s/it]

v3 maliyeti: $0.609


puan  uydurma  kural_ihlali
hakem_surum model                                    
v2-olculu   gpt-4.1-mini  0.93     0.50          0.63
            gpt-6-luna    1.40     0.27          0.30
v3          gpt-4.1-mini  1.23     0.33          0.40
            gpt-6-luna    1.37     0.27          0.33


Puanı değişen: 10 / 60


,,v2-olculu,v3,v3_gerekce
id,model,,,
D1,gpt-6-luna,2,1,Kısa selam veriyor ve araç kullanmıyor; ancak Steam oyunları hakkında hangi konularda yardımcı olabileceğini belirtmiyor.
K1,gpt-4.1-mini,1,2,Genel olumlu oy oranı doğru verilmiş ve oyunun beğenilmesiyle DLC fiyatlarına yönelik eleştiriler arasındaki ayrım açıklanmış. Yorumların örneklem olduğu belirtilmiş ve yasaklanan tavsiyeler aktarılmamış.
K3,gpt-4.1-mini,1,2,"Puan oyun_bul çıktısına, şikâyetler ise olumsuz yorum örneklerine dayanıyor. Yanıt yorumların örneklemden alındığını da belirtiyor."
R1,gpt-4.1-mini,1,2,"Genel puan doğru yuvarlanmış, yorumlar özetle modunda alınmış ve belirtilen olumlu ve olumsuz yönler araç çıktılarıyla destekleniyor."
R5,gpt-4.1-mini,1,2,Türkçe yorumlar dil='tr' filtresiyle alınmış ve olumlu ile olumsuz görüşler doğru özetlenmiş. Genel Steam puanı ve oy sayısı da oyun_bul çıktısına dayanıyor.
R6,gpt-4.1-mini,0,1,"8,5 saat araç çıktısında yorumcu medyan süresidir; bunun ortalama bitirme süresi olduğu desteklenmiyor. Yorum özeti ve olumlu oran ise araç çıktılarına dayanıyor."
S1,gpt-4.1-mini,1,2,Puan ve oy sayısı araç çıktısına dayanıyor; yüzde ve oy sayısı makul biçimde yuvarlanmış.
S2,gpt-4.1-mini,1,2,SQL sayımı 82 sonucunu veriyor ve cevap veri kümesinin dönemini doğru belirtiyor.
T1,gpt-4.1-mini,1,2,Olumlu oy oranı ve oy sayısı oyun_bul çıktısıyla destekleniyor; yorum verisinin bulunmadığı doğru biçimde belirtilmiş. Yorum uydurulmamış ve gerekli araçlar kullanılmış.


model,gpt-4.1-mini,gpt-6-luna
kategori,,
diger,1.67,1.33
karma,1.00,1.40
oneri,1.00,1.60
rag,1.50,0.67
sql,1.00,1.83
tuzak,1.40,1.40


## LLM hakem · gpt-6.1-terra
- Hakeme verilen: soru, kontrol, beklenen araç, dondurulmuş referans, agent'ın araç çağrıları + çıktıları (15.000 kar.), cevap, agent'ın sistem talimatı. JSON çıktı: puan 0/1/2, uydurma, kural_ihlali, gerekçe.
- Sürümler:
  - v1: araç çıktılarını 4.000 karakterde kesiyordu → görmediği yorumlara "uydurma" dedi → 15.000.
  - v2 (ölçülü): yan iddia uydurmaysa 1, ana iddia uydurmaysa 0 (uydurma bayrağı sıkı kalır).
  - v3: sistematik hatalar düzeltildi: talimattaki bilgi uydurma değil, puan oyun_bul'dan verilebilir, yuvarlama serbest, biçim kusuru puan düşürmez.
- v2 → v3: 10/60 puan değişti; 9'u mini'ye haksız verilen cezalardı. Gerçek hatalar değişmedi.
- Maliyet: 60 puanlama ≈ $0,6.

| Model (1. tekrar, v3) | Puan | Uydurma | Kural ihlali |
|---|---|---|---|
| gpt-4.1-mini | 1,23 | %33 | %40 |
| gpt-6-luna | 1,37 | %27 | %33 |

| Kategori | mini | Luna |
|---|---|---|
| sql | 1,00 | 1,83 |
| oneri | 1,00 | 1,60 |
| karma | 1,00 | 1,40 |
| tuzak | 1,40 | 1,40 |
| rag | 1,50 | 0,67 |
| diger | 1,67 | 1,33 |

- Mini'nin gerçek hataları: S3 (100 oy eşiği), S4 (indirimli fiyat), S6/O1 (gereksiz filtre, korkuda çocuk filtresi), O4/T5 (0 satır → "yok", kural 9), K2 (çeviri memnuniyeti uydurma), R6 (medyan oynama süresi = bitirme süresi).
- Luna'nın tek sistematik hatası: R1/R2/R5/K5'te yorum_ara yerine oyun_ara (mod parametresiyle) → yorumsuz cevap.
- Hakem gürültüsü: aynı cevap (D1 Luna) iki sürümde farklı puan aldı → küçük farkları yorumlama.

In [22]:
# 08 · Hata mesajı düzeltmesi: test + Luna'yı 30 soruda yeniden çalıştır (deney etiketiyle)
from src import agent_graf

print(agent_graf.arac_cagir("oyun_ara", '{"mod": "ozetle", "app_id": 1086940}'))

DENEY = "v4+ipucu"
MODEL, TEKRAR = "gpt-6-luna", 1

def yapilanlar():
    if not SONUC_DOSYA.exists():
        return set()
    with open(SONUC_DOSYA, encoding="utf-8") as f:
        return {(k["id"], k["tekrar"], k["model"], k["metin_modu"], k.get("deney", "v4")) for k in map(json.loads, f)}

bitti = yapilanlar()
isler = [(s, t) for t in range(1, TEKRAR + 1) for s in SORULAR
         if (s["id"], t, MODEL, ayarlar.METIN_MODU, DENEY) not in bitti]
print(f"\nModel: {MODEL} · deney: {DENEY} · yapılacak: {len(isler)}")
for s, t in tqdm(isler):
    try:
        kayit = bir_calistirma(s, t)
    except Exception as e:
        kayit = {"id": s["id"], "kategori": s["kategori"], "tekrar": t, "model": MODEL,
                 "metin_modu": ayarlar.METIN_MODU, "soru": s["soru"], "cevap": None,
                 "zaman": datetime.now().isoformat(timespec="seconds"), "hata": f"{type(e).__name__}: {e}"}
    kayit["deney"] = DENEY
    with open(SONUC_DOSYA, "a", encoding="utf-8") as f:
        f.write(json.dumps(kayit, ensure_ascii=False) + "\n")

tum = pd.read_json(SONUC_DOSYA, lines=True)
tum["deney"] = tum["deney"].fillna("v4") if "deney" in tum else "v4"
tum = tum[tum["metin_modu"] == "gizle"].copy()
tum["arac_dogru"] = tum.apply(arac_dogru, axis=1)
tum["hatalar"] = tum.apply(lambda r: [c for c in ciktilar(r) if c.startswith("HATA")], axis=1)

luna = tum[(tum["model"] == "gpt-6-luna") & (tum["tekrar"] == 1)]
display(luna.groupby("deney").agg(arac_dogru=("arac_dogru", "mean"), arac_hatali=("hatalar", lambda x: (x.str.len() > 0).sum()),
                                  tur=("tur", "mean"), sure=("sure", "mean"), maliyet=("maliyet_usd", "sum")).round(3))
display(luna.pivot_table(index="kategori", columns="deney", values="arac_dogru", aggfunc="mean").round(2))
display(luna[luna["id"].isin(["R1", "R2", "R5", "K5"])].pivot_table(index="id", columns="deney", values="araclar", aggfunc="first"))

HATA: yanlış parametre (oyun_araci() got an unexpected keyword argument 'mod'). 'oyun_ara' aracının parametreleri: istek, yetiskin_haric. Verdiğin parametreler (app_id, mod) 'yorum_ara' aracına ait; muhtemelen 'yorum_ara' aracını çağırmak istedin.

Model: gpt-6-luna · deney: v4+ipucu · yapılacak: 30


100%|██████████| 30/30 [03:55<00:00,  7.84s/it]


,arac_dogru,arac_hatali,tur,sure,maliyet
deney,,,,,
v4,0.833,3,3.700,7.30,0.059
v4+ipucu,0.900,1,4.233,7.75,0.071


deney,v4,v4+ipucu
kategori,,
diger,0.67,0.67
karma,0.80,1.00
oneri,1.00,1.00
rag,0.50,0.67
sql,1.00,1.00
tuzak,1.00,1.00


deney,v4,v4+ipucu
id,,
K5,"[oyun_bul({""isim"":""Half-Life: Alyx""}), oyun_ara({""istek"":""Half-Life: Alyx VR gözlüğü olmadan oynanabilir mi? Oyuncu yorumlarında VR gereksinimi ve VR'sız oynama deneyimi"",""yetiskin_haric"":true}), oyun_bul({""isim"":""Half-Life: Alyx""}), oyun_bul({""isim"":""Half-Life: Alyx""}), oyun_bul({""isim"":""Half-Life: Alyx""}), oyun_bul({""isim"":""Half-Life: Alyx""}), oyun_bul({""isim"":""Half-Life: Alyx""}), oyun_bul({""isim"":""Half-Life: Alyx""})]","[oyun_bul({""isim"":""Half-Life: Alyx""}), oyun_ara({""istek"":""Half-Life: Alyx VR gözlüğü olmadan oynanabilir mi? VR'sız oynama desteği ve oyuncu deneyimi"",""yetiskin_haric"":true}), oyun_bul({""isim"":""Half-Life: Alyx""}), oyun_bul({""isim"":""Half-Life: Alyx""}), oyun_bul({""isim"":""Half-Life: Alyx""}), sql_calistir({""sorgu"":""SELECT o.ad, o.olumlu_oran, o.oy_sayisi, o.yorum_sayisi, o.yorum_kapsami, EXISTS (SELECT 1 FROM oyun_kategori k WHERE k.app_id = o.app_id AND k.kategori = 'VR Only') AS vr_only, EXISTS (SELECT 1 FROM oyun_kategori k WHERE k.app_id = o.app_id AND k.kategori = 'VR Supported') AS vr_supported FROM oyunlar o WHERE o.app_id = 546560;""}), oyun_ara({""istek"":""Half-Life: Alyx VR gözlüğü olmadan oynama, VR'sız mod veya gözlüksüz oynanabilirlik"",""yetiskin_haric"":true}), oyun_bul({""isim"":""Half-Life: Alyx""})]"
R1,"[oyun_bul({""isim"":""Baldur's Gate 3""}), oyun_ara({""mod"":""ozetle"",""app_id"":1086940}), oyun_ara({""mod"":""ozetle"",""app_id"":1086940}), sql_calistir({""sorgu"":""SELECT app_id, ad, olumlu_oran, oy_sayisi, yorum_sayisi, yorum_kapsami FROM oyunlar WHERE app_id = 1086940""})]","[oyun_bul({""isim"":""Baldur's Gate 3""}), sql_calistir({""sorgu"": ""SELECT app_id, ad, olumlu_oran, oy_sayisi, yorum_sayisi, yorum_kapsami FROM oyunlar WHERE app_id = 1086940;""}), yorum_ara({""mod"": ""ozetle"", ""app_id"": 1086940})]"
R2,"[oyun_bul({""isim"":""Elden Ring""}), oyun_ara({""mod"":""ozetle"",""app_id"":1245620}), oyun_ara({""mod"":""ozetle"",""app_id"":1245620}), oyun_ara({""mod"":""ozetle"",""app_id"":1245620}), oyun_ara({""istek"":""Elden Ring oyuncularının en çok eleştirdiği yönler""}), oyun_ara({""istek"":""Elden Ring oyuncularının en çok eleştirdiği yönler"",""yetiskin_haric"":true}), oyun_ara({""istek"":""Elden Ring oyuncularının en çok eleştirdiği yönler"",""yetiskin_haric"":true}), oyun_ara({""istek"":""Elden Ring oyuncularının en çok eleştirdiği yönler"",""yetiskin_haric"":true})]","[oyun_bul({""isim"":""Elden Ring""}), oyun_ara({""mod"":""ozetle"",""app_id"":1245620}), oyun_ara({""istek"":""Elden Ring oyuncularının en çok eleştirdiği yönler"",""yetiskin_haric"":true}), oyun_bul({""isim"":""Elden Ring""}), oyun_ara({""mod"":""ozetle"",""app_id"":1245620}), oyun_ara({""istek"":""Elden Ring oyuncularının eleştirdiği şeyler"",""yetiskin_haric"":true}), oyun_bul({""isim"":""Elden Ring""}), oyun_ara({""istek"":""Elden Ring hakkında oyuncu yorumlarının özeti, özellikle olumsuz eleştiriler"",""yetiskin_haric"":true})]"
R5,"[oyun_bul({""isim"":""Mount & Blade II: Bannerlord""}), oyun_ara({""mod"":""ozetle"",""app_id"":261550}), oyun_ara({""mod"":""ozetle"",""app_id"":261550}), sql_calistir({""sorgu"":""SELECT app_id, ad, olumlu_oran, oy_sayisi, yorum_sayisi, yorum_kapsami, tr_yorum_sayisi, chroma_yorum_sayisi FROM oyunlar WHERE app_id = 261550""}), oyun_ara({""istek"":""Mount & Blade II: Bannerlord oyuncuların genel olarak sevdiği ve eleştirdiği yönler""}), oyun_ara({""istek"":""Mount & Blade II: Bannerlord hakkında Türkçe oyuncu yorumları: neden seviyorlar ve nelerden şikâyet ediyorlar""}), oyun_ara({""istek"":""Mount & Blade II: Bannerlord Türkçe dilindeki oyuncu yorumlarında oyunun sevilen ve eleştirilen yanları""}), oyun_ara({""istek"":""Mount & Blade II: Bannerlord oyuncu yorumlarının özeti""})]","[oyun_bul({""isim"":""Mount & Blade II: Bannerlord""}), oyun_bul({""isim"":""Mount & Blade II: Bannerlord""}), oyun_bul({""isim"":""Mount & Blade II: Bannerlord""}), oyun_bul({""isim"":""Mount & Blade II: Bannerlor

In [23]:
import importlib, json
from src import talimat, rag_araci, araclar, agent_graf
for modul in [talimat, rag_araci, araclar, agent_graf]:
    importlib.reload(modul)
from src.agent_graf import calistir

print("talimat uzunluğu:", len(talimat.SISTEM_TALIMATI))
print("araçlar:", list(araclar.ARAC_FONKSIYONLARI))
print("modelin gördüğü yerde 'oyun_ara' kaldı mı:",
      "oyun_ara" in talimat.SISTEM_TALIMATI or "oyun_ara" in json.dumps(araclar.ARAC_TANIMLARI, ensure_ascii=False))

# 1) Tekrar engeli: sahte konuşma
cagri1 = {"id": "c1", "type": "function", "function": {"name": "oyun_bul", "arguments": '{"isim": "Stray"}'}}
cagri2 = {"id": "c2", "type": "function", "function": {"name": "oyun_bul", "arguments": '{ "isim":"Stray" }'}}
cagri3 = {"id": "c3", "type": "function", "function": {"name": "oyun_bul", "arguments": '{"isim": "Stray 2"}'}}
durum = {"tur": 2, "mesajlar": [
    {"role": "assistant", "tool_calls": [cagri1]},
    {"role": "tool", "tool_call_id": "c1", "content": "(önceki sonuç)"},
    {"role": "assistant", "tool_calls": [cagri2, cagri3]}]}
sonuc = agent_graf.arac_dugumu(durum)
for k, m in zip(sonuc["iz"], sonuc["mesajlar"]):
    print(f"\n{k['ad']}({k['argumanlar']}) → tekrar={k['tekrar']}")
    print(m["content"])

# 2) Luna, sorun çıkan üç soru
SORULAR = json.loads(open("../reports/08_soru_seti.json", encoding="utf-8").read())
for s in [x for x in SORULAR if x["id"] in ("R2", "R5", "K5")]:
    r = calistir(s["soru"], oturum=f"test-v5-{s['id']}", model="gpt-6-luna")
    print("\n" + "=" * 80)
    print(s["id"], "|", s["soru"])
    print("tur:", r["tur"], "| süre:", r["sure"])
    for a in [a for a in r["iz"] if a["tip"] == "arac"]:
        print(f"  {a['ad']}({a['argumanlar']}) tekrar={a['tekrar']} hata={a['hata']}")
    print("CEVAP:\n" + (r["cevap"] or ""))

talimat uzunluğu: 3703
araçlar: ['sql_calistir', 'oyun_bul', 'yorum_ara', 'oyun_oner']
modelin gördüğü yerde 'oyun_ara' kaldı mı: False

oyun_bul({ "isim":"Stray" }) → tekrar=True
TEKRAR: Bu çağrıyı (aynı araç, aynı argümanlar) bu konuşmada zaten yaptın; sonucu yukarıdaki mesajlarda. Aynı çağrıyı tekrarlama: farklı bir araç ya da argüman dene veya elindeki bilgiyle cevap ver.

oyun_bul({"isim": "Stray 2"}) → tekrar=False
'Stray 2' için adı en çok benzeyen 5 oyun (benzerlik = ad benzerliği 0-100, kesin eşleşme anlamına gelmez; olumlu_oran = Steam oylarından genel puan, 0-1, boşsa oy yok):
app_id,ad,oy_sayisi,olumlu_oran,benzerlik
1332010,Stray,146316,0.973,91
1566990,-A-,28,0.679,90
1577470,Straya,77,0.883,83
1920780,Stray Gods: The Roleplaying Musical,2965,0.925,82
2901140,Stray Cats in Cozy Town,854,0.982,82
Kural: Sorulan oyunu ad ve oy sayısına bakarak seç. Adaylardan hiçbiri sorulan oyun değilse (farklı bir oyun, Playtest, araç veya kart oyunu) oyunun veri setinde olmadığını söyle.

### v5: tekrar engeli + oyun_oner adı
- **Tekrar engeli (agent_graf.py):** aynı araç + aynı argüman (JSON sıralı karşılaştırma, boşluk farkı önemsiz) ikinci kez istenirse araç çalışmaz, modele "TEKRAR" mesajı döner. İzde `tekrar=True` işareti tutulur. Sahte testte aynı çağrı yakalandı, farklı argüman normal çalıştı.
- **Ad değişikliği:** model `oyun_ara`'yı yorum aracı sanıyordu → modelin gördüğü ad `oyun_oner` oldu (iç fonksiyon aynı). Talimat 3702 → 3703 karakter, eski ad hiçbir yerde kalmadı.
- **Luna, sorunlu 3 soru (1'er kez):**
  - R2: artık `yorum_ara ozetle` çağırıyor, şikâyetler referansla uyumlu (takılma, 16:9, anti-cheat). Eskiden "yorumlara erişemiyorum" diyordu.
  - R5: yorumlara ulaştı ama `dil=tr` vermedi; genel özeti "Türkçe örneklem" diye sundu → kayma.
  - K5: döngü yok, cevap doğru ama `oyun_oner` gereksiz çağrıldı, SQL'deki "VR Only" kategorisine bakılmadı.
- Bu üç çalıştırmada tekrar engeli devreye girmedi; etkisi 30 soruluk ölçümde görülecek.

In [24]:
# 08 · v5 (tekrar engeli + oyun_oner): mini ve Luna 30×1
from src.agent_graf import calistir, GRAF
from src.talimat import SISTEM_TALIMATI

DENEY, TEKRAR = "v5", 1
ESKI_AD = {"oyun_oner": "oyun_ara"}   # soru setindeki beklenen araç eski adla yazılı

def arac_dogru(r):
    cagrilan = {ESKI_AD.get(a, a) for a in r["arac_adlari"]}
    bek = beklenen[r["id"]]
    if not bek:
        return len(cagrilan) == 0
    return set(bek) <= cagrilan and not (cagrilan & set(YASAK.get(r["id"], [])))

def bir_calistirma(s, tekrar):
    oturum = f"08-{s['id']}-{tekrar}-{datetime.now():%H%M%S}"
    if s["onceki"]:
        calistir(s["onceki"], oturum=oturum, model=MODEL)
    r = calistir(s["soru"], oturum=oturum, model=MODEL)
    gf, cf = FIYAT[MODEL]
    return {"id": s["id"], "kategori": s["kategori"], "tekrar": tekrar, "model": MODEL,
            "metin_modu": ayarlar.METIN_MODU, "talimat_kar": len(SISTEM_TALIMATI),
            "soru": s["soru"], "cevap": r["cevap"],
            "arac_adlari": [a["ad"] for a in r["iz"] if a["tip"] == "arac"], "araclar": r["araclar"],
            "arac_hata": sum(a.get("hata", False) for a in r["iz"]),
            "tekrar_engel": sum(a.get("tekrar", False) for a in r["iz"]),
            "tur": r["tur"], "sure": r["sure"], "girdi_token": r["girdi_token"], "cikti_token": r["cikti_token"],
            "maliyet_usd": round((r["girdi_token"] * gf + r["cikti_token"] * cf) / 1e6, 5),
            "arac_ciktilari": arac_ciktilari(oturum),
            "zaman": datetime.now().isoformat(timespec="seconds"), "hata": None}

print(f"talimat: {len(SISTEM_TALIMATI)} kar. · METIN_MODU: {ayarlar.METIN_MODU} · deney: {DENEY}")
for MODEL in ["gpt-4.1-mini", "gpt-6-luna"]:
    bitti = yapilanlar()
    isler = [(s, t) for t in range(1, TEKRAR + 1) for s in SORULAR
             if (s["id"], t, MODEL, ayarlar.METIN_MODU, DENEY) not in bitti]
    print(f"\nModel: {MODEL} · yapılacak: {len(isler)}")
    for s, t in tqdm(isler):
        try:
            kayit = bir_calistirma(s, t)
        except Exception as e:
            kayit = {"id": s["id"], "kategori": s["kategori"], "tekrar": t, "model": MODEL,
                     "metin_modu": ayarlar.METIN_MODU, "soru": s["soru"], "cevap": None,
                     "zaman": datetime.now().isoformat(timespec="seconds"), "hata": f"{type(e).__name__}: {e}"}
        kayit["deney"] = DENEY
        with open(SONUC_DOSYA, "a", encoding="utf-8") as f:
            f.write(json.dumps(kayit, ensure_ascii=False) + "\n")

# --- v4 → v5 karşılaştırma (sadece tekrar 1) ---
tum = pd.read_json(SONUC_DOSYA, lines=True)
tum["deney"] = tum["deney"].fillna("v4")
tum["tekrar_engel"] = tum["tekrar_engel"].fillna(0) if "tekrar_engel" in tum else 0
tum = tum[(tum["metin_modu"] == "gizle") & (tum["tekrar"] == 1)].copy()
tum["arac_dogru"] = tum.apply(arac_dogru, axis=1)
tum["hatalar"] = tum.apply(lambda r: [c for c in ciktilar(r) if c.startswith("HATA")], axis=1)

print("\n== Model × deney ==")
display(tum.groupby(["model", "deney"]).agg(
    n=("id", "size"), hatali_calistirma=("hata", lambda x: x.notna().sum()),
    arac_dogru=("arac_dogru", "mean"), arac_hatali=("hatalar", lambda x: (x.str.len() > 0).sum()),
    tekrar_engel=("tekrar_engel", "sum"), tur=("tur", "mean"), sure=("sure", "mean"),
    maliyet=("maliyet_usd", "sum")).round(3))

print("\n== Kategori bazında araç doğruluğu ==")
display(tum.pivot_table(index="kategori", columns=["model", "deney"], values="arac_dogru", aggfunc="mean").round(2))

v5 = tum[tum["deney"] == "v5"]
print("\n== v5: beklenen araç tutmayanlar ==")
display(v5[~v5["arac_dogru"]].assign(beklenen=lambda d: d["id"].map(beklenen))[["id", "model", "beklenen", "arac_adlari"]])

print("\n== v5: tekrar engeli devreye girenler ==")
for _, r in v5[v5["tekrar_engel"] > 0].iterrows():
    print(f"[{r['id']}] {r['model']} · engel: {int(r['tekrar_engel'])} · tur: {r['tur']} · çağrılar: {r['araclar']}")

print("\n== v5: araç hataları ==")
for _, r in v5[v5["hatalar"].str.len() > 0].iterrows():
    print(f"\n[{r['id']}] {r['model']} · çağrılar: {r['araclar']}")
    for h in r["hatalar"]:
        print("   ", h)

talimat: 3703 kar. · METIN_MODU: gizle · deney: v5

Model: gpt-4.1-mini · yapılacak: 30


100%|██████████| 30/30 [02:01<00:00,  4.03s/it]



Model: gpt-6-luna · yapılacak: 30


100%|██████████| 30/30 [03:04<00:00,  6.13s/it]


== Model × deney ==


n  hatali_calistirma  arac_dogru  arac_hatali  \
model        deney                                                      
gpt-4.1-mini v4        30                  0       0.967            0   
             v5        30                  0       0.933            1   
gpt-6-luna   v4        30                  0       0.833            3   
             v4+ipucu  30                  0       0.900            1   
             v5        30                  0       0.933            0   

                       tekrar_engel    tur   sure  maliyet  
model        deney                                          
gpt-4.1-mini v4                 0.0  2.567  4.063    0.154  
             v5                 0.0  2.600  3.910    0.156  
gpt-6-luna   v4                 0.0  3.700  7.300    0.059  
             v4+ipucu           0.0  4.233  7.750    0.071  
             v5                11.0  3.633  6.000    0.058


== Kategori bazında araç doğruluğu ==


model    gpt-4.1-mini      gpt-6-luna               
deney              v4   v5         v4 v4+ipucu    v5
kategori                                            
diger             1.0  1.0       0.67     0.67  0.67
karma             0.8  0.8       0.80     1.00  1.00
oneri             1.0  0.8       1.00     1.00  1.00
rag               1.0  1.0       0.50     0.67  0.83
sql               1.0  1.0       1.00     1.00  1.00
tuzak             1.0  1.0       1.00     1.00  1.00


== v5: beklenen araç tutmayanlar ==


,id,model,beklenen,arac_adlari
166,K5,gpt-4.1-mini,"[oyun_bul, sql_calistir]","[oyun_bul, yorum_ara]"
169,O3,gpt-4.1-mini,"[oyun_ara, sql_calistir]",[sql_calistir]
189,R4,gpt-6-luna,"[oyun_bul, yorum_ara]","[oyun_bul, oyun_bul, sql_calistir, oyun_bul, oyun_bul, oyun_bul, oyun_bul, oyun_bul]"
209,D3,gpt-6-luna,[yorum_ara],"[oyun_bul, oyun_bul, yorum_ara]"



== v5: tekrar engeli devreye girenler ==
[S3] gpt-6-luna · engel: 1 · tur: 4 · çağrılar: ['sql_calistir({"sorgu":"SELECT o.app_id, o.ad, o.olumlu_oran, o.oy_sayisi FROM oyunlar o WHERE o.olumlu_oran IS NOT NULL AND o.oy_sayisi >= 1000 AND o.yetiskin = 0 AND o.iskelet = 0 AND EXISTS (SELECT 1 FROM oyun_etiket e WHERE e.app_id = o.app_id AND e.etiket = \'Horror\') AND EXISTS (SELECT 1 FROM oyun_dil d WHERE d.app_id = o.app_id AND d.dil = \'Turkish\') ORDER BY o.olumlu_oran DESC, o.oy_sayisi DESC LIMIT 20"})', 'sql_calistir({"sorgu":"SELECT o.app_id, o.ad, o.olumlu_oran, o.oy_sayisi FROM oyunlar o WHERE o.olumlu_oran IS NOT NULL AND o.oy_sayisi >= 1000 AND o.yetiskin = 0 AND o.iskelet = 0 AND EXISTS (SELECT 1 FROM oyun_etiket e WHERE e.app_id = o.app_id AND e.etiket = \'Horror\') AND EXISTS (SELECT 1 FROM oyun_dil d WHERE d.app_id = o.app_id AND d.dil = \'Turkish\') ORDER BY o.olumlu_oran DESC, o.oy_sayisi DESC LIMIT 20"})', 'sql_calistir({"sorgu":"SELECT o.app_id, o.ad, o.olumlu_oran, o

### v5 sonuçları (30×1, tekrar 1 ile kıyas)
| | v4 | v4+ipucu | v5 |
|---|---|---|---|
| Luna araç doğruluğu | %83 | %90 | **%93** |
| Luna RAG | %50 | %67 | **%83** |
| Luna araç hatası | 3 | 1 | **0** |
| Luna süre / maliyet | 7,3 sn / $0,059 | 7,8 sn / $0,071 | **6,0 sn / $0,058** |
| mini araç doğruluğu | %97 | – | %93 |

- **Ad değişikliği (oyun_ara → oyun_oner) Luna'nın asıl sorununu çözdü:** yanlış araç + `mod` hatası tamamen kalktı.
- **Tekrar engeli** 6 soruda 11 kez devreye girdi. Çoğu zararsız (aynı SQL'in tekrarı, aynı turda çift oyun_bul, D3'te hafızadaki oyunu yeniden arama).
- **Kalan sorun, R4:** Luna "TEKRAR" mesajını yok sayıp oyun_bul'u 5 kez daha istedi, tur sınırına kadar yorum_ara'ya geçmedi. Engel aracı durduruyor, modelin isteğini durduramıyor.
- mini'deki %97 → %93 düşüşü tek soru farkı (O3'te oyun_oner'siz SQL); gürültü düzeyinde.

In [25]:
# 08 · Hakem v3: v4+ipucu ve v5 cevaplarını puanla, v4 ile kıyasla
ESKI_LISTE = "oyun_ara (tarife göre oyun)"
if "oyun_oner" not in HAKEM_TALIMATI:
    assert ESKI_LISTE in HAKEM_TALIMATI
    HAKEM_TALIMATI = HAKEM_TALIMATI.replace(ESKI_LISTE, "oyun_oner (tarife göre oyun; eski kayıtlarda adı oyun_ara)")
HAKEM_SURUM = "v3"
ANAHTAR = ["id", "model", "tekrar", "metin_modu", "deney"]

def hakem_yapilanlar():
    if not HAKEM_DOSYA.exists():
        return set()
    with open(HAKEM_DOSYA, encoding="utf-8") as f:
        return {(k["id"], k["model"], k["tekrar"], k["metin_modu"], k.get("deney", "v4")) for k in map(json.loads, f)
                if k["hakem_model"] == HAKEM_MODEL and k["hakem_surum"] == HAKEM_SURUM}

bitti = hakem_yapilanlar()
secilen = tum[tum["deney"].isin(["v4+ipucu", "v5"])]
isler = [r for _, r in secilen.iterrows() if tuple(r[a] for a in ANAHTAR) not in bitti]
print(f"Hakem: {HAKEM_MODEL} · {HAKEM_SURUM} · puanlanacak: {len(isler)}")
for r in tqdm(isler):
    try:
        h = hakem_puanla(r)
    except Exception as e:
        h = {"puan": None, "uydurma": None, "kural_ihlali": None, "gerekce": f"HAKEM HATASI: {e}", "hakem_maliyet": 0}
    kayit = {**{a: (int(r[a]) if a == "tekrar" else r[a]) for a in ANAHTAR},
             "hakem_model": HAKEM_MODEL, "hakem_surum": HAKEM_SURUM, **h}
    with open(HAKEM_DOSYA, "a", encoding="utf-8") as f:
        f.write(json.dumps(kayit, ensure_ascii=False) + "\n")

hk = pd.read_json(HAKEM_DOSYA, lines=True)
hk["deney"] = hk["deney"].fillna("v4")
hk = hk[(hk["hakem_model"] == HAKEM_MODEL) & (hk["hakem_surum"] == HAKEM_SURUM)]
print(f"Bu turun hakem maliyeti: ${hk[hk['deney'] != 'v4']['hakem_maliyet'].sum():.3f}")

m = tum.merge(hk[ANAHTAR + ["puan", "uydurma", "kural_ihlali", "gerekce"]], on=ANAHTAR, how="inner")

print("\n== Model × deney (hakem v3) ==")
display(m.groupby(["model", "deney"])[["puan", "uydurma", "kural_ihlali"]].mean().round(2))

print("\n== Kategori bazında puan ==")
display(m.pivot_table(index="kategori", columns=["model", "deney"], values="puan", aggfunc="mean").round(2))

print("\n== Soru bazında puan ==")
display(m.pivot_table(index="id", columns=["model", "deney"], values="puan", aggfunc="first")
         .reindex([s["id"] for s in SORULAR]))

print("\n== v4 → v5 puanı değişenler ==")
for model in ["gpt-4.1-mini", "gpt-6-luna"]:
    a = m[(m["model"] == model) & (m["deney"] == "v4")].set_index("id")
    b = m[(m["model"] == model) & (m["deney"] == "v5")].set_index("id")
    for i in [s["id"] for s in SORULAR]:
        if i in a.index and i in b.index and a.loc[i, "puan"] != b.loc[i, "puan"]:
            print(f"\n[{i}] {model}: {a.loc[i, 'puan']} → {b.loc[i, 'puan']}")
            print(f"   v4 araçlar: {a.loc[i, 'arac_adlari']}")
            print(f"   v5 araçlar: {b.loc[i, 'arac_adlari']}")
            print(f"   v4 hakem: {a.loc[i, 'gerekce']}")
            print(f"   v5 hakem: {b.loc[i, 'gerekce']}")

Hakem: gpt-6.1-terra · v3 · puanlanacak: 90


100%|██████████| 90/90 [06:51<00:00,  4.58s/it]

Bu turun hakem maliyeti: $0.943

== Model × deney (hakem v3) ==


puan  uydurma  kural_ihlali
model        deney                                
gpt-4.1-mini v4        1.23     0.33          0.40
             v5        1.07     0.47          0.50
gpt-6-luna   v4        1.37     0.27          0.33
             v4+ipucu  1.30     0.20          0.43
             v5        1.43     0.17          0.37


== Kategori bazında puan ==


model    gpt-4.1-mini       gpt-6-luna               
deney              v4    v5         v4 v4+ipucu    v5
kategori                                             
diger            1.67  1.67       1.33     1.33  1.00
karma            1.00  0.80       1.40     1.60  1.60
oneri            1.00  0.80       1.60     1.40  1.20
rag              1.50  0.83       0.67     0.50  1.17
sql              1.00  1.33       1.83     1.83  1.83
tuzak            1.40  1.20       1.40     1.20  1.60


== Soru bazında puan ==


model gpt-4.1-mini    gpt-6-luna            
deney           v4 v5         v4 v4+ipucu v5
id                                          
S1               2  2          2        2  2
S2               2  2          2        2  2
S3               0  0          1        1  1
S4               0  0          2        2  2
S5               2  2          2        2  2
S6               0  2          2        2  2
R1               2  2          1        1  2
R2               1  1          0        0  2
R3               2  1          1        1  1
R4               1  1          2        1  0
R5               2  0          0        0  2
R6               1  0          0        0  0
K1               2  2          2        2  2
K2               0  0          2        2  2
K3               2  1          2        2  2
K4               1  1          1        1  1
K5               0  0          0        1  1
O1               1  1          1        0  0
O2               1  1          1        2  1
O3               2  1          2        2  2
O4               0  1          2        2  2
O5               1  0          2        1  1
T1               2  2          2        2  2
T2               2  2          2        1  2
T3               2  1          1        1  2
T4               1  1          1        1  1
T5               0  0          1        1  1
D1               2  2          1        1  2
D2               2  2          2        2  0
D3               1  1          1        1  1


== v4 → v5 puanı değişenler ==

[S6] gpt-4.1-mini: 0 → 2
   v4 araçlar: ['sql_calistir']
   v5 araçlar: ['sql_calistir']
   v4 hakem: Gereksiz yetişkin ve iskelet filtreleri nedeniyle verilen sayı 3.888; istenen toplam ise 3.895'tir. Cevap araç çıktısına dayanmaktadır.
   v5 hakem: Online Co-op kategorisi doğru sorgulanmış ve referansla aynı olan 3.895 oyun sayısı verilmiş.

[R3] gpt-4.1-mini: 2 → 1
   v4 araçlar: ['oyun_bul', 'oyun_bul', 'yorum_ara']
   v5 araçlar: ['oyun_bul', 'oyun_bul', 'yorum_ara']
   v4 hakem: Cevabın ana iddiası, geceyi korkutucu ve kalp atışlarını hızlandıran bir kâbus olarak tanımlayan ilgili yorumlarla destekleniyor. Zayıf eşleşme veya ironik yorum kanıt olarak kullanılmamış.
   v5 hakem: Gecelerin korkutucu olduğu ilgili yorumlarla destekleniyor; ancak zombi saldırılarının özellikle korkutucu bulunduğu ve gecenin daha tehlikeli olduğu görünen yorumlarda belirtilmiyor. Bu desteklenmeyen yan iddialar veriye dayanma kuralını da ihlal ediyor.

[R5] gpt-4.1-mini

### Hakem v3: v4 → v5 cevap kalitesi (30×1)
| | puan | uydurma | kural ihlali |
|---|---|---|---|
| mini v4 | 1,23 | %33 | %40 |
| mini v5 | 1,07 | %47 | %50 |
| Luna v4 | 1,37 | %27 | %33 |
| Luna v4+ipucu | 1,30 | %20 | %43 |
| Luna v5 | **1,43** | **%17** | %37 |

- **Düzeltme hedefini vurdu:** Luna R1, R2, R5 → 2 (yorum_ara artık çağrılıyor; araç yolu değişti, şans değil).
- **Kalan yapısal sorun:** R4, tekrar engeline rağmen oyun_bul döngüsü (puan 0).
- **Gürültü:** mini'de kod değişmediği hâlde 9/30 soruda puan değişti (S6 0→2, R5 2→0...). Luna'da D2 (kod yazdı), O1, O5 düşüşleri de koddan bağımsız.
- **Ders:** temperature 0'da bile soruların ~1/3'ü çalıştırmadan çalıştırmaya değişiyor → 30×1 ile ±0,15 fark anlamsız. 09'da her soru ≥3 tekrar, ortalama + yayılım raporlanacak.

In [26]:
# 08 · Gürültü ölçümü: mini v4 tekrar 2–3 hakem puanı, 3 tekrar karşılaştırması
hepsi = pd.read_json(SONUC_DOSYA, lines=True)
hepsi["deney"] = hepsi["deney"].fillna("v4")
mini3 = hepsi[(hepsi["model"] == "gpt-4.1-mini") & (hepsi["deney"] == "v4") & (hepsi["metin_modu"] == "gizle")].copy()

bitti = hakem_yapilanlar()
isler = [r for _, r in mini3[mini3["tekrar"].isin([2, 3])].iterrows() if tuple(r[a] for a in ANAHTAR) not in bitti]
print(f"Hakem: {HAKEM_MODEL} · {HAKEM_SURUM} · puanlanacak: {len(isler)}")
for r in tqdm(isler):
    try:
        h = hakem_puanla(r)
    except Exception as e:
        h = {"puan": None, "uydurma": None, "kural_ihlali": None, "gerekce": f"HAKEM HATASI: {e}", "hakem_maliyet": 0}
    kayit = {**{a: (int(r[a]) if a == "tekrar" else r[a]) for a in ANAHTAR},
             "hakem_model": HAKEM_MODEL, "hakem_surum": HAKEM_SURUM, **h}
    with open(HAKEM_DOSYA, "a", encoding="utf-8") as f:
        f.write(json.dumps(kayit, ensure_ascii=False) + "\n")

hk = pd.read_json(HAKEM_DOSYA, lines=True)
hk["deney"] = hk["deney"].fillna("v4")
hk = hk[(hk["hakem_model"] == HAKEM_MODEL) & (hk["hakem_surum"] == HAKEM_SURUM)]
print(f"Bu turun hakem maliyeti: ${sum(1 for _ in isler) and hk[(hk['model'] == 'gpt-4.1-mini') & (hk['deney'] == 'v4') & (hk['tekrar'] > 1)]['hakem_maliyet'].sum():.3f}")
g = mini3.merge(hk[ANAHTAR + ["puan", "uydurma", "kural_ihlali", "gerekce"]], on=ANAHTAR, how="inner")

print("\n== Tekrar bazında ortalama (aynı model, aynı kod) ==")
display(g.groupby("tekrar")[["puan", "uydurma", "kural_ihlali"]].mean().round(2))

p = g.pivot_table(index="id", columns="tekrar", values="puan", aggfunc="first").reindex([s["id"] for s in SORULAR])
p["ortalama"] = p[[1, 2, 3]].mean(axis=1).round(2)
p["fark"] = p[[1, 2, 3]].max(axis=1) - p[[1, 2, 3]].min(axis=1)
print("\n== Soru bazında 3 tekrar ==")
display(p)
print(f"\n3 tekrarda aynı puan: {(p['fark'] == 0).sum()} / 30 · 1 puan oynayan: {(p['fark'] == 1).sum()} · 0↔2 oynayan: {(p['fark'] == 2).sum()}")
print(f"3 tekrarın ortalaması: {p['ortalama'].mean():.2f}")
display(p.join(pd.Series({s['id']: s['kategori'] for s in SORULAR}, name="kategori")).groupby("kategori")["fark"].mean().round(2))

print("\n== 0↔2 oynayan sorular: araçlar ve hakem gerekçesi ==")
for i in p[p["fark"] == 2].index:
    print(f"\n[{i}] {SORU[i]['soru']}")
    for _, r in g[g["id"] == i].sort_values("tekrar").iterrows():
        print(f"   t{r['tekrar']} · puan {r['puan']} · araçlar {r['arac_adlari']}")
        print(f"      hakem: {r['gerekce']}")

Hakem: gpt-6.1-terra · v3 · puanlanacak: 60


100%|██████████| 60/60 [04:36<00:00,  4.61s/it]

Bu turun hakem maliyeti: $0.603

== Tekrar bazında ortalama (aynı model, aynı kod) ==


,puan,uydurma,kural_ihlali
tekrar,,,
1,1.23,0.33,0.40
2,1.17,0.43,0.30
3,1.10,0.37,0.37



== Soru bazında 3 tekrar ==


tekrar,1,2,3,ortalama,fark
id,,,,,
S1,2,2,2,2.00,0
S2,2,2,2,2.00,0
S3,0,0,2,0.67,2
S4,0,0,0,0.00,0
S5,2,2,2,2.00,0
S6,0,0,0,0.00,0
R1,2,2,2,2.00,0
R2,1,1,1,1.00,0
R3,2,1,1,1.33,1



3 tekrarda aynı puan: 19 / 30 · 1 puan oynayan: 9 · 0↔2 oynayan: 2
3 tekrarın ortalaması: 1.17


kategori
diger    0.33
karma    0.40
oneri    0.60
rag      0.67
sql      0.33
tuzak    0.20
Name: fark, dtype: float64


== 0↔2 oynayan sorular: araçlar ve hakem gerekçesi ==

[S3] Türkçe destekleyen en yüksek puanlı korku oyunları hangileri?
   t1 · puan 0 · araçlar ['sql_calistir']
      hakem: Gerekli 1.000 oy eşiği yerine 100 oy kullanılarak sıralamanın başına yanlış oyunlar yerleştirilmiş ve eşik cevapta belirtilmemiş. Türkçe desteğinin geliştirici tarafından belirtildiği ve puanların güvenilir olduğu iddiaları araç çıktısında desteklenmiyor.
   t2 · puan 0 · araçlar ['sql_calistir']
      hakem: En az 1.000 oy şartı yerine 100 oy eşiği kullanılmış; bu nedenle listenin başında koşulu karşılamayan oyunlar yer alıyor ve sıralama yanlış.
   t3 · puan 2 · araçlar ['sql_calistir']
      hakem: Oyunlar en az 1.000 oy eşiğiyle olumlu oranına göre sıralanmış ve listelenen ilk oyunlar referansla örtüşüyor. SQL sonucundaki tekrarlar çıkarılarak sunulan sekiz oyun, belirli bir liste uzunluğu istemeyen soruyu karşılıyor.

[R5] Türk oyuncular Mount & Blade II: Bannerlord hakkında ne diyor?
   t1 · puan 2 · araç

### Gürültü ölçümü: mini v4, 3 tekrar (hakem v3)
| tekrar | puan | uydurma | kural ihlali |
|---|---|---|---|
| 1 | 1,23 | %33 | %40 |
| 2 | 1,17 | %43 | %30 |
| 3 | 1,10 | %37 | %37 |
| **ortalama** | **1,17** | | |

- Aynı model + aynı kod → 30 soruluk ortalama **0,13'lük aralıkta** oynuyor. mini v5'in 1,07'si bu aralığın kenarında; düşüş değil gürültü.
- Soru bazında: **19/30 kararlı**, 9 soru ±1, 2 soru 0↔2 (S3, R5). En kararsız: RAG (0,67), öneri (0,60); en kararlı: tuzak (0,20).
- Oynamanın kaynağı **model**, hakem değil: S3'te talimattaki "oy ≥ 1000" kuralına 3'te 1 uyuyor; R5'te `dil=tr`'yi 3'te 2 veriyor.
- **Kalıcı hatalar** (her tekrarda 0): S4 indirimli fiyat, K5 araçsız cevap, O4/T5 0 satır → "yok" (kural 9), S6 gereksiz filtre (4'te 3).
- Luna v5 (1,43) − mini (1,17) = 0,26 > gürültü aralığı → Luna büyük ihtimalle daha iyi; kesin sonuç 09'da 3 tekrarla.
- **09 kararı:** her soru 3 tekrar, model puanı = 3 tekrarın ortalaması; 0,15'ten küçük farklar "eşit" sayılacak.

In [27]:
# 08 · METIN_MODU: gizle vs kes (mini + Luna 30×1, hakem v3)
from src import sql_araci

TANIM = sql_araci.SQL_ARAC_TANIMLARI[0]["function"]
ESKI_ACIKLAMA = TANIM["description"]
KES_NOTU = f"bu araçta en fazla {sql_araci.KESME_UZUNLUGU} karakterde kesilmiş olarak döner (içerik özeti için yorum_ara kullan)."
assert ESKI_ACIKLAMA.count("bu araçta KAPALI.") == 2

DENEY, TEKRAR = "v5", 1
try:
    ayarlar.METIN_MODU = sql_araci.METIN_MODU = "kes"
    TANIM["description"] = ESKI_ACIKLAMA.replace("bu araçta KAPALI.", KES_NOTU)
    print(sql_araci.sql_calistir("SELECT metin FROM yorumlar WHERE app_id = 1332010 LIMIT 1"))
    for MODEL in ["gpt-4.1-mini", "gpt-6-luna"]:
        bitti = yapilanlar()
        isler = [(s, t) for t in range(1, TEKRAR + 1) for s in SORULAR
                 if (s["id"], t, MODEL, ayarlar.METIN_MODU, DENEY) not in bitti]
        print(f"\nModel: {MODEL} · METIN_MODU: {ayarlar.METIN_MODU} · yapılacak: {len(isler)}")
        for s, t in tqdm(isler):
            try:
                kayit = bir_calistirma(s, t)
            except Exception as e:
                kayit = {"id": s["id"], "kategori": s["kategori"], "tekrar": t, "model": MODEL,
                         "metin_modu": ayarlar.METIN_MODU, "soru": s["soru"], "cevap": None,
                         "zaman": datetime.now().isoformat(timespec="seconds"), "hata": f"{type(e).__name__}: {e}"}
            kayit["deney"] = DENEY
            with open(SONUC_DOSYA, "a", encoding="utf-8") as f:
                f.write(json.dumps(kayit, ensure_ascii=False) + "\n")
finally:
    ayarlar.METIN_MODU = sql_araci.METIN_MODU = "gizle"
    TANIM["description"] = ESKI_ACIKLAMA
print(f"\nGeri alındı: METIN_MODU = {sql_araci.METIN_MODU} · açıklamada KAPALI: {'KAPALI' in TANIM['description']}")

# --- hakem ---
hepsi = pd.read_json(SONUC_DOSYA, lines=True)
hepsi["deney"] = hepsi["deney"].fillna("v4")
v5 = hepsi[(hepsi["deney"] == "v5") & (hepsi["tekrar"] == 1)].copy()
bitti = hakem_yapilanlar()
isler = [r for _, r in v5[v5["metin_modu"] == "kes"].iterrows() if tuple(r[a] for a in ANAHTAR) not in bitti]
print(f"\nHakem: {HAKEM_MODEL} · {HAKEM_SURUM} · puanlanacak: {len(isler)}")
for r in tqdm(isler):
    try:
        h = hakem_puanla(r)
    except Exception as e:
        h = {"puan": None, "uydurma": None, "kural_ihlali": None, "gerekce": f"HAKEM HATASI: {e}", "hakem_maliyet": 0}
    kayit = {**{a: (int(r[a]) if a == "tekrar" else r[a]) for a in ANAHTAR},
             "hakem_model": HAKEM_MODEL, "hakem_surum": HAKEM_SURUM, **h}
    with open(HAKEM_DOSYA, "a", encoding="utf-8") as f:
        f.write(json.dumps(kayit, ensure_ascii=False) + "\n")

hk = pd.read_json(HAKEM_DOSYA, lines=True)
hk["deney"] = hk["deney"].fillna("v4")
hk = hk[(hk["hakem_model"] == HAKEM_MODEL) & (hk["hakem_surum"] == HAKEM_SURUM)]
k = v5.merge(hk[ANAHTAR + ["puan", "uydurma", "kural_ihlali", "gerekce"]], on=ANAHTAR, how="inner")
k["arac_dogru"] = k.apply(arac_dogru, axis=1)
k["metne_baktı"] = k["araclar"].map(lambda a: any(("metin" in c or "aciklama" in c or "oyun_hakkinda" in c)
                                                  for c in a if c.startswith("sql_calistir")))

print("\n== gizle vs kes (v5, tekrar 1) ==")
display(k.groupby(["model", "metin_modu"]).agg(
    puan=("puan", "mean"), uydurma=("uydurma", "mean"), kural_ihlali=("kural_ihlali", "mean"),
    arac_dogru=("arac_dogru", "mean"), metne_bakan=("metne_baktı", "sum"),
    tur=("tur", "mean"), sure=("sure", "mean"), girdi_token=("girdi_token", "mean"), maliyet=("maliyet_usd", "sum")).round(3))
display(k.pivot_table(index="kategori", columns=["model", "metin_modu"], values="puan", aggfunc="mean").round(2))

print("\n== kes modunda SQL ile metne bakan çalıştırmalar ==")
for _, r in k[(k["metin_modu"] == "kes") & k["metne_baktı"]].iterrows():
    g = k[(k["id"] == r["id"]) & (k["model"] == r["model"]) & (k["metin_modu"] == "gizle")]
    gp = g["puan"].iloc[0] if len(g) else None
    print(f"\n[{r['id']}] {r['model']} · puan kes {r['puan']} / gizle {gp} · {r['soru']}")
    for c in r["araclar"]:
        print("   ", c)
    print(f"   hakem: {r['gerekce']}")

1 satır, 0.01 sn
metin
在这里养了一只猫猫，路过的朋友可以摸摸她一赞摸一次　　　　 　　 ＿＿　　　 　　／＞　　フ　　　 　　| 　_　 _ l　 　　 　／` ミ＿xノ　　 　 /　　　 　 |　　　 /　 ヽ　　 ﾉ　 　 │　　|　|　|　／￣|　　 |　|　|　| (￣ヽ＿_ヽ_)__)　＼二つ ​


Model: gpt-4.1-mini · METIN_MODU: kes · yapılacak: 30


100%|██████████| 30/30 [02:06<00:00,  4.22s/it]



Model: gpt-6-luna · METIN_MODU: kes · yapılacak: 30


100%|██████████| 30/30 [03:12<00:00,  6.43s/it]



Geri alındı: METIN_MODU = gizle · açıklamada KAPALI: True

Hakem: gpt-6.1-terra · v3 · puanlanacak: 60


100%|██████████| 60/60 [04:08<00:00,  4.15s/it]


== gizle vs kes (v5, tekrar 1) ==


puan  uydurma  kural_ihlali  arac_dogru  \
model        metin_modu                                             
gpt-4.1-mini gizle       1.067    0.467         0.500       0.933   
             kes         1.067    0.400         0.433       0.967   
gpt-6-luna   gizle       1.433    0.167         0.367       0.933   
             kes         1.500    0.067         0.333       0.900   

                         metne_bakan    tur   sure  girdi_token  maliyet  
model        metin_modu                                                   
gpt-4.1-mini gizle                 0  2.600  3.910    12181.933    0.156  
             kes                   0  2.633  4.157    12444.167    0.159  
gpt-6-luna   gizle                 0  3.633  6.000    17882.167    0.058  
             kes                   1  3.500  6.310    17195.167    0.056

model      gpt-4.1-mini       gpt-6-luna      
metin_modu        gizle   kes      gizle   kes
kategori                                      
diger              1.67  2.00       1.00  1.00
karma              0.80  1.00       1.60  1.60
oneri              0.80  1.00       1.20  2.00
rag                0.83  0.83       1.17  0.67
sql                1.33  1.00       1.83  2.00
tuzak              1.20  1.00       1.60  1.60


== kes modunda SQL ile metne bakan çalıştırmalar ==

[T4] gpt-6-luna · puan kes 1 / gizle 1 · 吞食一统中原Tunshi  unifying plains nasıl bir oyun, alınır mı?
    oyun_bul({"isim":"吞食一统中原Tunshi unifying plains"})
    sql_calistir({"sorgu": "SELECT app_id, ad, olumlu_oran, oy_sayisi, yorum_sayisi, yorum_kapsami, fiyat, normal_fiyat, kisa_aciklama, oyun_hakkinda FROM oyunlar WHERE app_id = 2002600"})
    yorum_ara({"mod": "ozetle", "app_id": 2002600})
   hakem: Az oy nedeniyle puanın güvenilirliğinin sınırlı olduğunu belirtiyor, ancak 16 yorumluk örneklemin az olduğunu açıkça söylemiyor.


### METIN_MODU: gizle vs kes (v5, 30×1, hakem v3)
| | puan | uydurma | araç doğruluğu | SQL'de metne bakan |
|---|---|---|---|---|
| mini gizle | 1,07 | %47 | %93 | – |
| mini kes | 1,07 | %40 | %97 | 0 |
| Luna gizle | 1,43 | %17 | %93 | – |
| Luna kes | 1,50 | %7 | %90 | 1 (T4, oyun açıklaması) |

- Kes modunda (açıklama "200 karakterde kesilmiş döner" diye güncellenerek) modeller **60 çalıştırmanın 1'inde** metin sütununa baktı, yorum metnine hiç. Davranış aynı → farklar gürültü (0,07 < 0,15).
- Kategori farkları (Luna öneri 1,20→2,00, RAG 1,17→0,67) metin kullanılmadan oluştu → tek çalıştırmada 5–6 soruluk kategori ortalaması güvenilmez.
- **Karar: gizle kalıyor.** Kalite eşit; gizle daha güvenli (SQL'de resim/kopya temizliği yok, kontrol sorgusunda ilk yorum ASCII kedi çıktı) ve engel modeli RAG aracına yönlendiriyor.
- Not: SQL araç açıklamasındaki "metin: KAPALI" moddan bağımsız sabit; kes seçilirse açıklama moda bağlanmalı.

In [28]:
# 08 · Elle puanlama dosyaları: okuma (md) + doldurma (csv), hakem puanı gizli
secim = hepsi[(hepsi["model"] == "gpt-6-luna") & (hepsi["deney"] == "v5") &
              (hepsi["metin_modu"] == "gizle") & (hepsi["tekrar"] == 1)].set_index("id")

satirlar = ["# 08 · Elle puanlama: gpt-6-luna v5 (hakem puanı gizli)\n",
            "Puan: **2** doğru ve eksiksiz · **1** ana cevap doğru ama eksik/küçük hata · **0** yanlış, uydurma ya da ciddi kural ihlali\n"]
for s in SORULAR:
    r = secim.loc[s["id"]]
    on = f"*(Önce sorulan: {s['onceki']})*  \n" if s["onceki"] else ""
    satirlar += [f"\n---\n\n## {s['id']} · {s['soru']}\n", on,
                 f"**Kontrol:** {s['kontrol']}  \n**Beklenen araç:** {s['beklenen_arac']}\n",
                 f"\n**Referans:**\n\n{s['referans_metin'] or '(yok)'}\n",
                 f"\n**Agent'ın araçları:** {r['arac_adlari']}\n",
                 f"\n**Cevap:**\n\n{r['cevap']}\n"]
OKUMA = Path("../reports/08_elle_puan_oku.md")
OKUMA.write_text("\n".join(satirlar), encoding="utf-8")

DOLDUR = Path("../reports/08_elle_puan.csv")
if DOLDUR.exists():
    print(f"{DOLDUR} zaten var, üzerine yazmadım (puanların silinmesin diye).")
else:
    pd.DataFrame({"id": [s["id"] for s in SORULAR], "kaan_puan": "", "kaan_not": ""}).to_csv(DOLDUR, index=False, encoding="utf-8")
print(f"Oku: {OKUMA}\nDoldur: {DOLDUR}")

Oku: ../reports/08_elle_puan_oku.md
Doldur: ../reports/08_elle_puan.csv


## 08 · Sonuç
- 30 soruluk set (6 kategori, dondurulmuş referanslar) + otomatik kontroller + LLM hakem (gpt-6.1-terra, talimat v3).
- **mini (baseline):** 3 tekrar ortalaması 1,17/2, sayı uydurma yok; kalıcı hatalar S4, K5, O4/T5 (kural 9), S6.
- **Luna v5:** 1,43/2, uydurma %17, araç doğruluğu %93, 6 sn, $0,058/30 soru. 17/30 tam doğru.
- **Yapısal düzeltmeler** (talimat kuralı yerine): hata mesajına ipucu, `oyun_ara → oyun_oner`, tekrar engeli. Asıl kazanç ad değişikliğinden geldi (Luna RAG %50 → %83). Kalan sorun: R4 döngüsü.
- **Gürültü:** aynı model + kod → ortalama 0,13 aralıkta oynuyor, soruların ~1/3'ü değişiyor. 09'da 3 tekrar, <0,15 fark = eşit.
- **METIN_MODU:** kes modunda modeller metne neredeyse hiç bakmadı → **gizle kalıyor**.
- Toplam maliyet ≈ $4,5. Maliyet hatalı tahmin tam emin değilim.